# Smart Cities, Cooler Futures

## A Data-Driven Framework for Urban Cooling Prioritisation

### Final analytical architecture

**Business question**

Which urban characteristics are associated with higher urban surface heat, where is cooling pressure greatest, and what cooling actions are supported by the evidence?

### Analytical logic

1. Prepare and validate the raw spatial datasets.
2. Use Land Surface Temperature (LST) as the main measurable heat outcome where spatial resolution permits.
3. Test explicit hypotheses before building any decision-support score.
4. Distinguish statistical evidence from descriptive city profiles.
5. Use only evidence-supported factors to inform prioritisation and cooling recommendations.
6. Treat city-level UHI as contextual evidence because the source provides one representative UHI value per city.

This notebook intentionally does not assume that a high normalised indicator is a proven heat driver.


# Smart Cities, Cooler Futures

## Urban Cooling Analysis

This project analyzes urban heat vulnerability across ten European cities and identifies which cities should be prioritised for urban cooling interventions.

### Selected Cities

- Berlin
- Athens
- Madrid
- Barcelona
- Milan
- Vienna
- Prague
- Budapest
- Lisbon
- Amsterdam

### Data Sources

1. Copernicus Urban Atlas 2021
2. Urban Atlas Street Tree Layer 2021
3. Population data (2021)
4. Urban Heat and Heatwave data (Map 2.5)
5. Copernicus IMDC imperviousness change data (2021–2024)

### Data Timeframe

We use 2021 as the baseline because it provides a consistent dataset across the main urban indicators.

We complement this baseline with 2021–2024 imperviousness change data to capture more recent urban changes that may influence future cooling priorities.

## 1. Project Setup

The project directories are defined first so that all datasets can be accessed consistently throughout the analysis.

### 1.1 Project Paths

The project directories are defined first so that all datasets can be accessed consistently throughout the analysis.

In [2]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

BASE_PATH = Path(
    "/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures"
)

DATA_PATH = BASE_PATH / "data_raw"

URBAN_ATLAS_PATH = DATA_PATH / "urban_atlas"
STREET_TREES_PATH = DATA_PATH / "street_trees"
POPULATION_PATH = DATA_PATH / "population"
HEAT_PATH = DATA_PATH / "heat"

print("Project:", BASE_PATH)
print("Data:", DATA_PATH)

Project: /Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures
Data: /Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw


### 1.2 Define the Building Data Path

The building dataset is included as an additional data source for the urban cooling analysis.

The GHS-OBAT R2024A building data is stored in country-level folders within the `data_raw/Building/` directory.

In [3]:
# Define the Building data path

BUILDING_PATH = DATA_PATH / "Building"

print("Building data directory:")
print(BUILDING_PATH)

Building data directory:
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/Building


### 1.3 Check Building Folder Contents

The Building data folder is checked to verify that the expected country-level datasets are available before starting the building analysis.

In [4]:
# Check Building country folders

print("Building folder contents:\n")

for item in sorted(BUILDING_PATH.iterdir()):
    if item.is_dir():
        print("-", item.name)

Building folder contents:

- GHS_OBAT_CSV_AUT_E2020_R2024A_V1_0
- GHS_OBAT_CSV_CZE_E2020_R2024A_V1_0
- GHS_OBAT_CSV_DEU_E2020_R2024A_V1_0
- GHS_OBAT_CSV_ESP_E2020_R2024A_V1_0
- GHS_OBAT_CSV_GRC_E2020_R2024A_V1_0
- GHS_OBAT_CSV_HUN_E2020_R2024A_V1_0
- GHS_OBAT_CSV_ITA_E2020_R2024A_V1_0
- GHS_OBAT_CSV_NLD_E2020_R2024A_V1_0
- GHS_OBAT_CSV_PRT_E2020_R2024A_V1_0


## 2. Urban Atlas

Urban Atlas 2021 provides detailed land use and land cover
information for the ten selected European cities.

The data is stored as FlatGeobuf (`.fgb`) files.

In [5]:
# Find Urban Atlas FGB files

fgb_files = sorted(URBAN_ATLAS_PATH.rglob("*.fgb"))

print(f"Found {len(fgb_files)} Urban Atlas files")

Found 10 Urban Atlas files


### 2.1 Urban Atlas Data Structure

Before combining the city datasets, we inspect one file
to understand the available variables and land-use classification.

In [6]:
# Inspect one Urban Atlas file

sample_gdf = gpd.read_file(fgb_files[0])

print("Shape:", sample_gdf.shape)
print("\nColumns:")
print(sample_gdf.columns.tolist())

display(sample_gdf.head())

Shape: (90323, 11)

Columns:
['country', 'fua_name', 'fua_code', 'code_2021', 'class_2021', 'prod_date', 'identifier', 'perimeter', 'area', 'comment', 'geometry']


,country,fua_name,fua_code,code_2021,class_2021,prod_date,identifier,perimeter,area,comment,geometry
0,AT,Wien,AT001L3,21000,Arable land (annual crops),2025-09,70626-AT001L3,584.488251,22755.177704,None,"MULTIPOLYGON (((4849627.919 2757717.629, 48496..."
1,AT,Wien,AT001L3,23000,Pastures,2025-09,84190-AT001L3,1805.337560,119901.961521,None,"MULTIPOLYGON (((4847684.504 2757187.15, 484768..."
2,AT,Wien,AT001L3,23000,Pastures,2025-09,84176-AT001L3,414.713551,10866.489328,None,"MULTIPOLYGON (((4847385.417 2757561.293, 48473..."
3,AT,Wien,AT001L3,23000,Pastures,2025-09,84186-AT001L3,780.743799,17642.653582,None,"MULTIPOLYGON (((4847588.642 2757102.584, 48475..."
4,AT,Wien,AT001L3,23000,Pastures,2025-09,83923-AT001L3,543.393359,12376.342939,None,"MULTIPOLYGON (((4841878.454 2755912.078, 48418..."


### 2.2 Combining Urban Atlas Data

The Urban Atlas datasets for the ten selected cities are combined
into a single GeoDataFrame.

The `city` column is retained to identify the origin of each record
and enable comparisons between cities.

In [7]:
# Combine Urban Atlas data for all cities

urban_atlas_data = []

for file in fgb_files:
    gdf = gpd.read_file(file)
    
    # Use the Functional Urban Area name as the city identifier
    gdf["city"] = gdf["fua_name"]
    
    urban_atlas_data.append(gdf)

urban_atlas = gpd.GeoDataFrame(
    pd.concat(urban_atlas_data, ignore_index=True),
    crs=urban_atlas_data[0].crs
)

print("Combined Urban Atlas shape:", urban_atlas.shape)
print("\nCities included:")
print(sorted(urban_atlas["city"].unique()))

Combined Urban Atlas shape: (790454, 12)

Cities included:
['Amsterdam', 'Athina', 'Barcelona', 'Berlin', 'Budapest', 'Lisboa', 'Madrid', 'Milano', 'Praha', 'Wien']


### 2.3 Urban Atlas Land-Use Classes

Urban Atlas classifies land use and land cover using standardized
codes for each Functional Urban Area.

We inspect the available classes before grouping them into broader
categories for the urban cooling analysis.

In [8]:
# Inspect Urban Atlas land-use classes

land_use_classes = (
    urban_atlas[["code_2021", "class_2021"]]
    .drop_duplicates()
    .sort_values("code_2021")
)

display(land_use_classes)

,code_2021,class_2021
122,11100,Continuous urban fabric (S.L. : > 80%)
94,11210,Discontinuous dense urban fabric (S.L. : 50% -...
164,11220,Discontinuous medium density urban fabric (S.L...
490,11230,Discontinuous low density urban fabric (S.L. :...
93,11240,Discontinuous very low density urban fabric (S...
345,11300,Isolated structures
5,12100,"Industrial, commercial, public, military and p..."
725,12210,Fast transit roads and associated land
1255,12220,Other roads and associated land
3618,12230,Railways and associated land


### 2.4 Green Space Classification

For the urban cooling analysis, Urban Atlas land-use classes are
grouped into broader categories.

Green-space areas include public, private, and unknown-access
green urban areas, as well as forests and natural herbaceous
vegetation.

These categories will be used to calculate the proportion of
green space within each city.

In [9]:
# Define Urban Atlas green-space classes

green_space_codes = [
    "14110",  # Green urban areas - Public access
    "14120",  # Green urban areas - Private access
    "14130",  # Green urban areas - Unknown access
    "31000",  # Forests
    "32000",  # Herbaceous vegetation associations
]

urban_atlas["green_space"] = urban_atlas["code_2021"].isin(
    green_space_codes
)

print("Green-space records:", urban_atlas["green_space"].sum())

Green-space records: 87994


### 2.4 Urban Atlas Code Validation

The Urban Atlas land-use codes are checked before defining the green-space classes.

In [10]:
# Check the data type and unique Urban Atlas codes

print("code_2021 data type:")
print(urban_atlas["code_2021"].dtype)

print("\nSample code values:")
print(urban_atlas["code_2021"].drop_duplicates().head(30).tolist())

code_2021 data type:
str

Sample code values:
['21000', '23000', '12100', '50000', '13100', '22000', '11240', '11210', '14200', '14110', '11100', '31000', '11220', '13400', '11300', '11230', '12210', '12220', '33000', '40000', '32000', '13300', '12230', '14130', '14120', '24000', '12400', '12300']


### 2.5 Green Space Area by City

The total area of identified green-space classes is calculated for each city.

In [11]:
# Calculate total green-space area by city

green_space_by_city = (
    urban_atlas[urban_atlas["green_space"]]
    .groupby("city")["area"]
    .sum()
    .sort_values(ascending=False)
)

display(green_space_by_city)

city
Berlin       3.639715e+09
Madrid       2.990688e+09
Wien         2.269757e+09
Praha        1.635195e+09
Lisboa       1.604253e+09
Budapest     1.553230e+09
Barcelona    1.277947e+09
Athina       9.276018e+08
Milano       3.092296e+08
Amsterdam    2.724769e+08
Name: area, dtype: float64

### 2.6 Area Unit Validation

The `area` field and geometry-derived area are compared to validate the measurement unit before converting areas to square kilometres.

In [12]:
# Check the area values and geometry CRS

print("CRS:", urban_atlas.crs)

print("\nArea statistics:")
print(urban_atlas["area"].describe())

CRS: EPSG:3035

Area statistics:
count    7.904540e+05
mean     6.853080e+04
std      9.202104e+05
min      1.857921e-01
25%      5.207567e+03
50%      1.141234e+04
75%      2.555906e+04
max      3.431194e+08
Name: area, dtype: float64


In [13]:
# Compare the area field with geometry-derived area

sample = urban_atlas.iloc[0]

print("Area field:", sample["area"])
print("Geometry area:", sample.geometry.area)

Area field: 22755.177704020094
Geometry area: 22755.177704020094


### 2.7 Green Space Area in km²

Urban Atlas area values are converted from square metres (m²) to square kilometres (km²) for easier interpretation and comparison.

In [14]:
# Calculate green-space area in square kilometres

green_space_by_city_km2 = (
    urban_atlas[urban_atlas["green_space"]]
    .groupby("city")["area"]
    .sum()
    .div(1_000_000)
    .sort_values(ascending=False)
)

display(green_space_by_city_km2)

city
Berlin       3639.715336
Madrid       2990.688215
Wien         2269.756597
Praha        1635.195480
Lisboa       1604.253378
Budapest     1553.230263
Barcelona    1277.947018
Athina        927.601791
Milano        309.229584
Amsterdam     272.476885
Name: area, dtype: float64

### 2.8 Green Space Percentage by City

To enable a fair comparison between cities of different sizes, we calculate the percentage of each city's Urban Atlas area classified as green space.

In [15]:
# Calculate total Urban Atlas area by city

total_area_by_city = (
    urban_atlas
    .groupby("city")["area"]
    .sum()
)

# Calculate green-space area by city

green_area_by_city = (
    urban_atlas[urban_atlas["green_space"]]
    .groupby("city")["area"]
    .sum()
)

# Calculate green-space percentage

green_space_percentage = (
    green_area_by_city
    .div(total_area_by_city)
    .mul(100)
    .sort_values(ascending=False)
)

display(green_space_percentage)

city
Barcelona    48.296473
Athina       47.222665
Berlin       40.156175
Madrid       37.972160
Lisboa       36.514981
Praha        28.370472
Budapest     24.274084
Wien         23.588846
Milano        9.927272
Amsterdam     8.189335
Name: area, dtype: float64

### 2.9 Urban Atlas Green Space Indicator

The green-space percentage is stored as a city-level indicator for comparison between cities.

In [16]:
# Create a city-level green-space indicator table

urban_atlas_indicators = (
    green_space_percentage
    .rename("green_space_pct")
    .reset_index()
)

urban_atlas_indicators

,city,green_space_pct
0,Barcelona,48.296473
1,Athina,47.222665
2,Berlin,40.156175
3,Madrid,37.972160
4,Lisboa,36.514981
5,Praha,28.370472
6,Budapest,24.274084
7,Wien,23.588846
8,Milano,9.927272
9,Amsterdam,8.189335


In [17]:
# Add green-space area in km²

urban_atlas_indicators["green_space_km2"] = (
    green_space_by_city_km2
    .reindex(urban_atlas_indicators["city"])
    .values
)

urban_atlas_indicators

,city,green_space_pct,green_space_km2
0,Barcelona,48.296473,1277.947018
1,Athina,47.222665,927.601791
2,Berlin,40.156175,3639.715336
3,Madrid,37.972160,2990.688215
4,Lisboa,36.514981,1604.253378
5,Praha,28.370472,1635.195480
6,Budapest,24.274084,1553.230263
7,Wien,23.588846,2269.756597
8,Milano,9.927272,309.229584
9,Amsterdam,8.189335,272.476885


## 3. Street Trees

Street tree data is used to complement the Urban Atlas green-space
indicator by providing information about trees located along urban
streets.

The ten selected European cities are analyzed consistently with
the Urban Atlas dataset.

### 3.1 Street Trees Data Structure

The Street Trees dataset provides street-tree information for the ten selected European cities.

The files are stored as FlatGeobuf (`.fgb`) files.

In [18]:
# Select one Street Trees FGB file for each city

street_tree_files = sorted(
    [
        file for file in STREET_TREES_PATH.rglob("*.fgb")
        if file.parent != STREET_TREES_PATH
    ]
)

print(f"Found {len(street_tree_files)} city-level Street Trees files:\n")

for file in street_tree_files:
    print(file)

Found 20 city-level Street Trees files:

/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/street_trees/Athina/CLMS_UA_STL_S2021_V005ha_EL001L1_ATHINA_03035_V01_R02_20251121.fgb
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/street_trees/Athina/CLMS_UA_UM_S2021_V005ha_EL001L1_ATHINA_03035_V01_R02_20251121.fgb
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/street_trees/Berlin/CLMS_UA_STL_S2021_V005ha_DE001L1_BERLIN_03035_V01_R02_20251121.fgb
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/street_trees/Berlin/CLMS_UA_UM_S2021_V005ha_DE001L1_BERLIN_03035_V01_R02_20251121.fgb
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/street_trees/CLMS_UA_STL_S2021_V005ha_CZ001L2_PRAHA_03035_V01_R02_20251121/CLMS_UA_STL_S2021_V005ha_CZ001L2_PRAHA_03035_V01_R02_20251121.fgb
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/street_trees/CLMS_UA_STL_S2021_V005ha_CZ

In [19]:
# Inspect one Street Trees file

sample_street_trees = gpd.read_file(street_tree_files[0])

print("Shape:", sample_street_trees.shape)

print("\nCRS:", sample_street_trees.crs)

print("\nColumns:")
print(sample_street_trees.columns.tolist())

display(sample_street_trees.head())

Shape: (25375, 7)

CRS: EPSG:3035

Columns:
['area', 'perimeter', 'country', 'fua_code', 'fua_name', 'STL', 'geometry']


,area,perimeter,country,fua_code,fua_name,STL,geometry
0,537.5,136.148276,EL,EL001L1,ATHINA,1,"MULTIPOLYGON (((5563503.876 1737212.827, 55634..."
1,600.0,123.027756,EL,EL001L1,ATHINA,1,"MULTIPOLYGON (((5563618.876 1737352.827, 55636..."
2,1362.5,163.087106,EL,EL001L1,ATHINA,1,"MULTIPOLYGON (((5562928.876 1737482.827, 55629..."
3,1225.0,172.503869,EL,EL001L1,ATHINA,1,"MULTIPOLYGON (((5562998.876 1737307.827, 55630..."
4,1187.5,146.534064,EL,EL001L1,ATHINA,1,"MULTIPOLYGON (((5562898.876 1737182.827, 55629..."


### 3.2 Street Trees File Inspection

Before combining the city datasets, we inspect one file to verify its structure, attributes, and coordinate reference system.

In [20]:
# Inspect one Street Trees file

sample_street_trees = gpd.read_file(street_tree_files[0])

print("Shape:", sample_street_trees.shape)
print("\nCRS:", sample_street_trees.crs)
print("\nColumns:")
print(sample_street_trees.columns.tolist())

display(sample_street_trees.head())

Shape: (25375, 7)

CRS: EPSG:3035

Columns:
['area', 'perimeter', 'country', 'fua_code', 'fua_name', 'STL', 'geometry']


,area,perimeter,country,fua_code,fua_name,STL,geometry
0,537.5,136.148276,EL,EL001L1,ATHINA,1,"MULTIPOLYGON (((5563503.876 1737212.827, 55634..."
1,600.0,123.027756,EL,EL001L1,ATHINA,1,"MULTIPOLYGON (((5563618.876 1737352.827, 55636..."
2,1362.5,163.087106,EL,EL001L1,ATHINA,1,"MULTIPOLYGON (((5562928.876 1737482.827, 55629..."
3,1225.0,172.503869,EL,EL001L1,ATHINA,1,"MULTIPOLYGON (((5562998.876 1737307.827, 55630..."
4,1187.5,146.534064,EL,EL001L1,ATHINA,1,"MULTIPOLYGON (((5562898.876 1737182.827, 55629..."


### 3.3 Combining Street Trees Data

The Street Trees datasets for the ten selected European cities are combined into a single GeoDataFrame.

The `fua_name` field is retained to identify the corresponding city for city-level aggregation and comparison.

In [21]:
# Combine Street Trees data for all ten cities

street_tree_data = []

for file in street_tree_files:
    gdf = gpd.read_file(file)
    street_tree_data.append(gdf)

street_trees = gpd.GeoDataFrame(
    pd.concat(street_tree_data, ignore_index=True),
    crs=street_tree_data[0].crs
)

print("Combined Street Trees shape:", street_trees.shape)

print("\nCities included:")
print(
    sorted(
        street_trees["fua_name"]
        .dropna()
        .astype(str)
        .str.upper()
        .unique()
    )
)

Combined Street Trees shape: (532898, 8)

Cities included:
['AMSTERDAM', 'ATHINA', 'BARCELONA', 'BERLIN', 'BUDAPEST', 'LISBOA', 'MADRID', 'MILANO', 'PRAHA', 'WIEN']


### 3.4 Street Tree Area by City

The total area of street-tree features is aggregated for each city.

This provides a city-level measure of street-tree coverage and complements the broader green-space indicator derived from Urban Atlas.

In [22]:
# Calculate total street-tree area by city

street_tree_area_by_city = (
    street_trees
    .groupby("fua_name")["area"]
    .sum()
    .sort_values(ascending=False)
)

display(street_tree_area_by_city)

fua_name
BERLIN       529384262.5
BUDAPEST     272653000.0
PRAHA        196415950.0
WIEN         185659150.0
MADRID       164719537.5
MILANO       149984612.5
AMSTERDAM    140381650.0
BARCELONA    101012187.5
LISBOA        81106900.0
ATHINA        63926837.5
Name: area, dtype: float64

### 3.5 Street Trees Area Validation

Before calculating city-level street-tree indicators, we validate the area measurements and coordinate reference system used by the Street Trees dataset.

In [23]:
# Validate Street Trees area measurements

print("CRS:", street_trees.crs)

print("\nArea statistics:")
print(street_trees["area"].describe())

# Compare the stored area with the geometry area for one feature
sample = street_trees.iloc[0]

print("\nSample area check:")
print("Area field:", sample["area"])
print("Geometry area:", sample.geometry.area)

CRS: EPSG:3035

Area statistics:
count    4.538990e+05
mean     4.153444e+03
std      6.928396e+04
min      5.000000e+02
25%      7.625000e+02
50%      1.250000e+03
75%      2.587500e+03
max      3.262756e+07
Name: area, dtype: float64

Sample area check:
Area field: 537.5
Geometry area: 537.5


### 3.6 Street Tree Area in km²

Street Trees area values are converted from square metres (m²) to square kilometres (km²) for easier interpretation and comparison.

In [24]:
# Calculate total street-tree area in square kilometres

street_tree_area_km2 = (
    street_trees
    .groupby("fua_name")["area"]
    .sum()
    .div(1_000_000)
    .sort_values(ascending=False)
)

display(street_tree_area_km2)

fua_name
BERLIN       529.384262
BUDAPEST     272.653000
PRAHA        196.415950
WIEN         185.659150
MADRID       164.719538
MILANO       149.984612
AMSTERDAM    140.381650
BARCELONA    101.012187
LISBOA        81.106900
ATHINA        63.926837
Name: area, dtype: float64

### 3.7 Street Tree Percentage by City

To enable a comparable assessment across cities of different sizes, street-tree area is expressed as a percentage of the corresponding Urban Atlas area.

In [25]:
# Calculate total Urban Atlas area by city in km²

total_area_km2 = (
    urban_atlas
    .groupby("city")["area"]
    .sum()
    .div(1_000_000)
)

# Calculate street-tree percentage

street_tree_percentage = (
    street_tree_area_km2
    .div(total_area_km2.rename(index=str.upper))
    .mul(100)
    .sort_values(ascending=False)
)

display(street_tree_percentage)

fua_name
BERLIN       5.840580
MILANO       4.814992
BUDAPEST     4.261056
AMSTERDAM    4.219192
BARCELONA    3.817476
PRAHA        3.407796
ATHINA       3.254409
MADRID       2.091410
WIEN         1.929496
LISBOA       1.846103
Name: area, dtype: float64

### 3.8 Urban Green and Street Tree Indicators

The Urban Atlas green-space and Street Tree indicators are combined into a single city-level table.

This table will serve as the basis for integrating the Population and Heat indicators.

In [26]:
# Create a city-level Street Tree indicator table

street_tree_indicators = (
    street_tree_percentage
    .rename("street_tree_pct")
    .reset_index()
    .rename(columns={"fua_name": "city"})
)

# Standardize city names for merging

urban_atlas_indicators["city"] = (
    urban_atlas_indicators["city"].str.upper()
)

street_tree_indicators["city"] = (
    street_tree_indicators["city"].str.upper()
)

# Merge Urban Atlas and Street Tree indicators

city_green_indicators = urban_atlas_indicators.merge(
    street_tree_indicators,
    on="city",
    how="inner"
)

display(city_green_indicators.sort_values("green_space_pct", ascending=False))

,city,green_space_pct,green_space_km2,street_tree_pct
0,BARCELONA,48.296473,1277.947018,3.817476
1,ATHINA,47.222665,927.601791,3.254409
2,BERLIN,40.156175,3639.715336,5.840580
3,MADRID,37.972160,2990.688215,2.091410
4,LISBOA,36.514981,1604.253378,1.846103
5,PRAHA,28.370472,1635.195480,3.407796
6,BUDAPEST,24.274084,1553.230263,4.261056
7,WIEN,23.588846,2269.756597,1.929496
8,MILANO,9.927272,309.229584,4.814992
9,AMSTERDAM,8.189335,272.476885,4.219192


### 3.9 Green and Street Tree Indicators

The combined city-level table summarizes two complementary urban vegetation indicators:

- `green_space_pct`: percentage of the Urban Atlas area classified as green space.
- `street_tree_pct`: percentage of the Urban Atlas area covered by Street Tree Layer features.

These indicators will later be combined with population and heat data to assess urban cooling conditions.

## 4. Population

Population data is used to provide demographic context for the urban cooling analysis.

The population indicator is aligned with the same ten selected European cities used in the Urban Atlas and Street Trees analysis.

### 4.1 Population Data Structure

Before calculating the population indicator, we inspect the available population files and identify the data needed for the analysis.

In [27]:
population_raster = (
    POPULATION_PATH
    / "JRC-ESTAT_Census_Population_2021_100m_rev0726.tif"
)

print("Population raster exists:", population_raster.exists())

Population raster exists: True


### 4.2 Population Raster Structure

The population dataset is provided as a 100 m raster grid for the 2021 census reference period.

We inspect the raster dimensions, coordinate reference system, resolution, and NoData value before extracting population by city.

In [28]:
import rasterio

population_raster = (
    POPULATION_PATH
    / "JRC-ESTAT_Census_Population_2021_100m_rev0726.tif"
)

with rasterio.open(population_raster) as src:
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Data type:", src.dtypes[0])
    print("NoData value:", src.nodata)

CRS: EPSG:3035
Width: 55620
Height: 44750
Resolution: (100.0, 100.0)
Bounds: BoundingBox(left=943000.0, bottom=941000.0, right=6505000.0, top=5416000.0)
Data type: int16
NoData value: -1.0


### 4.3 Population Extraction by City

The 100 m population raster is spatially matched with the
Functional Urban Area boundaries from Urban Atlas.

For each city, the population values of raster cells within
the FUA boundary are summed to obtain a city-level population
indicator for the 2021 reference period.

In [29]:
# 4.3 Population Extraction by City

from rasterio.features import rasterize
from rasterio.windows import from_bounds

population_by_city = {}

with rasterio.open(population_raster) as src:

    for city in sorted(urban_atlas["city"].unique()):

        print(f"Processing {city}...")

        city_gdf = urban_atlas.loc[
            urban_atlas["city"] == city,
            ["geometry"]
        ].copy()

        # Simplify geometries to match the 100 m raster resolution
        city_gdf["geometry"] = city_gdf.geometry.simplify(
            tolerance=50,
            preserve_topology=True
        )

        # Bounding box
        minx, miny, maxx, maxy = city_gdf.total_bounds

        window = from_bounds(
            minx, miny, maxx, maxy,
            transform=src.transform
        ).round_offsets().round_lengths()

        population_array = src.read(
            1,
            window=window
        )

        window_transform = src.window_transform(window)

        # Rasterize the simplified Urban Atlas polygons
        city_mask = rasterize(
            [(geom, 1) for geom in city_gdf.geometry if not geom.is_empty],
            out_shape=population_array.shape,
            transform=window_transform,
            fill=0,
            dtype="uint8"
        )

        valid = (
            (city_mask == 1) &
            (population_array != src.nodata)
        )

        population_by_city[city] = population_array[valid].sum()

        print(f"Population: {population_by_city[city]:,.0f}")

population_by_city = pd.Series(
    population_by_city,
    name="population_2021"
)

display(
    population_by_city.sort_values(ascending=False)
)

Processing Amsterdam...


/opt/miniconda3/lib/python3.13/site-packages/shapely/constructive.py:1177: RuntimeWarning: invalid value encountered in simplify_preserve_topology
  return lib.simplify_preserve_topology(geometry, tolerance, **kwargs)


Population: 2,682,860
Processing Athina...


/opt/miniconda3/lib/python3.13/site-packages/shapely/constructive.py:1177: RuntimeWarning: invalid value encountered in simplify_preserve_topology
  return lib.simplify_preserve_topology(geometry, tolerance, **kwargs)


Population: 3,418,170
Processing Barcelona...


...

/opt/miniconda3/lib/python3.13/site-packages/shapely/constructive.py:1177: RuntimeWarning: invalid value encountered in simplify_preserve_topology
  return lib.simplify_preserve_topology(geometry, tolerance, **kwargs)


Population: 2,088,432
Processing Wien...


/opt/miniconda3/lib/python3.13/site-packages/shapely/constructive.py:1177: RuntimeWarning: invalid value encountered in simplify_preserve_topology
  return lib.simplify_preserve_topology(geometry, tolerance, **kwargs)


Population: 2,840,721


Madrid       6507965
Barcelona    4767556
Milano       4609706
Berlin       4595309
Athina       3418170
Budapest     2842187
Wien         2840721
Lisboa       2743699
Amsterdam    2682860
Praha        2088432
Name: population_2021, dtype: int64

### 4.4 City-Level Population Indicator

The extracted 2021 population values are converted into a
city-level table and will later be combined with the urban
green-space and street-tree indicators.

In [30]:
# Create city-level population indicator table

population_indicators = (
    population_by_city
    .rename("population_2021")
    .reset_index()
    .rename(columns={"index": "city"})
)

# Standardize city names for merging
population_indicators["city"] = (
    population_indicators["city"]
    .str.upper()
)

display(
    population_indicators
    .sort_values("population_2021", ascending=False)
)

,city,population_2021
6,MADRID,6507965
2,BARCELONA,4767556
7,MILANO,4609706
3,BERLIN,4595309
1,ATHINA,3418170
4,BUDAPEST,2842187
9,WIEN,2840721
5,LISBOA,2743699
0,AMSTERDAM,2682860
8,PRAHA,2088432


### 4.5 Combined City Indicators

The population indicator is combined with the Urban Atlas
green-space and Street Tree indicators to create a unified
city-level dataset for the ten selected cities.

In [31]:
# Merge population with green-space and street-tree indicators

city_indicators = city_green_indicators.merge(
    population_indicators,
    on="city",
    how="inner"
)

display(
    city_indicators
    .sort_values("population_2021", ascending=False)
)

,city,green_space_pct,green_space_km2,street_tree_pct,population_2021
3,MADRID,37.972160,2990.688215,2.091410,6507965
0,BARCELONA,48.296473,1277.947018,3.817476,4767556
8,MILANO,9.927272,309.229584,4.814992,4609706
2,BERLIN,40.156175,3639.715336,5.840580,4595309
1,ATHINA,47.222665,927.601791,3.254409,3418170
6,BUDAPEST,24.274084,1553.230263,4.261056,2842187
7,WIEN,23.588846,2269.756597,1.929496,2840721
4,LISBOA,36.514981,1604.253378,1.846103,2743699
9,AMSTERDAM,8.189335,272.476885,4.219192,2682860
5,PRAHA,28.370472,1635.195480,3.407796,2088432


## 5. Urban Heat

The heat dataset is based on EEA Map 2.5 and provides two
complementary indicators related to urban heat:

- urban heat island intensity
- heatwave-related exposure

The two indicators will be inspected and aggregated at city level
before being combined with the environmental and population data.

### 5.1 Heat Data Structure

Before extracting the heat indicators, we inspect the available
heat layers and their spatial attributes.

The heat dataset contains two complementary layers:

- `Map_2_5_intensity`
- `Map_2_5_heatwaves`

### 5.2 Heat Layer Inspection

The EEA Map 2.5 dataset contains two spatial heat layers:

- `Map_2_5_intensity`: urban heat island intensity
- `Map_2_5_heatwaves`: heatwave-related information

We inspect both layers to identify their attributes, coordinate
reference system, and spatial structure before aggregating the
heat indicators by city.

In [32]:
# Define the two heat shapefiles

HEAT_DATA_PATH = HEAT_PATH / "Map2.5-120001" / "data"

heatwaves_path = HEAT_DATA_PATH / "Map_2_5_heatwaves.shp"
intensity_path = HEAT_DATA_PATH / "Map_2_5_intensity.shp"

print("Heatwaves exists:", heatwaves_path.exists())
print("Intensity exists:", intensity_path.exists())

Heatwaves exists: True
Intensity exists: True


In [33]:
# Read both heat layers

heatwaves = gpd.read_file(heatwaves_path)
intensity = gpd.read_file(intensity_path)

print("Heatwaves shape:", heatwaves.shape)
print("Intensity shape:", intensity.shape)

print("\nHeatwaves CRS:", heatwaves.crs)
print("Intensity CRS:", intensity.crs)

print("\nHeatwaves columns:")
print(heatwaves.columns.tolist())

print("\nIntensity columns:")
print(intensity.columns.tolist())

Heatwaves shape: (154, 6)
Intensity shape: (100, 6)

Heatwaves CRS: EPSG:3035
Intensity CRS: EPSG:3035

Heatwaves columns:
['OBJECTID', 'Id', 'hw85_20_52', 'Shape_Leng', 'Shape_Area', 'geometry']

Intensity columns:
['OBJECTID_1', 'OBJECTID', 'Column1', 'Column2', 'Column3', 'geometry']


### 5.3 Heat Attribute Inspection

The heat layers contain several attribute fields. Their values are
inspected before selecting the variables used in the city-level
heat indicators.

This step helps ensure that the analysis uses the correct heat
intensity and heatwave variables from the EEA Map 2.5 dataset.

In [34]:
# Inspect Heatwaves attributes

print("Heatwaves:")
display(heatwaves.head())

print("\nHeatwaves descriptive statistics:")
display(heatwaves.describe())

Heatwaves:


,OBJECTID,Id,hw85_20_52,Shape_Leng,Shape_Area,geometry
0,1,1,5.0,538399.184415,1.204751e+10,"POLYGON ((5692093.438 6008708.517, 5645357.918..."
1,2,2,1.5,539171.641354,1.204753e+10,"POLYGON ((5737530.076 6047571.912, 5692093.438..."
2,3,3,1.5,677258.814874,2.671955e+10,"POLYGON ((5645357.918 5971175.419, 5692093.438..."
3,4,4,0.0,567855.596588,1.603769e+10,"POLYGON ((4575723.655 4923741.954, 4556303.062..."
4,5,5,0.0,877865.234738,4.805947e+10,"POLYGON ((6173889.73 5578598.116, 5988898.265 ..."



Heatwaves descriptive statistics:


,OBJECTID,Id,hw85_20_52,Shape_Leng,Shape_Area
count,154.000000,154.000000,154.000000,1.540000e+02,1.540000e+02
mean,77.500000,82.214286,2.444805,1.209784e+06,7.833404e+10
std,44.600075,45.854126,2.130974,1.608517e+06,1.528157e+11
min,1.000000,1.000000,0.000000,5.383992e+05,1.204751e+10
25%,39.250000,44.250000,1.000000,6.894657e+05,2.821925e+10
50%,77.500000,82.500000,1.500000,7.395485e+05,3.371919e+10
75%,115.750000,120.750000,3.875000,1.208514e+06,7.227208e+10
max,154.000000,162.000000,9.500000,1.550216e+07,1.271231e+12


In [35]:
# Inspect Heat Intensity values

display(intensity.head(10))

print("\nUnique values:")
for column in ["Column1", "Column2", "Column3"]:
    print(f"\n{column}:")
    print(intensity[column].unique())

,OBJECTID_1,OBJECTID,Column1,Column2,Column3,geometry
0,1,1,-0.477337,38.362244,1.071040,POINT Z (3402258.625 1760840.363 1.071)
1,2,2,4.896728,52.364805,1.446170,POINT Z (3973698.674 3262772.191 1.446)
2,3,3,4.402000,51.246000,1.243220,POINT Z (3930553.314 3141075.188 1.243)
3,4,4,23.728056,37.983889,2.101180,POINT Z (5528466.128 1764761.749 2.101)
4,5,5,2.132641,41.384404,1.370220,POINT Z (3661587.218 2065827.861 1.37)
5,6,6,16.838012,41.122724,0.723279,POINT Z (4896838.966 2028577.647 0.723)
6,7,7,7.593117,47.561912,1.461090,POINT Z (4139786.084 2719423.963 1.461)
7,8,8,20.402604,44.813520,1.481330,POINT Z (5142343.508 2468663.147 1.481)
8,9,9,13.410963,52.518918,1.730470,POINT Z (4552446.382 3273166.94 1.731)
9,10,10,-2.935181,43.290576,0.953347,POINT Z (3274279.494 2332983.716 0.953)



Unique values:

Column1:
[ -0.477337   4.896728   4.402     23.728056   2.132641  16.838012
   7.593117  20.402604  13.410963  -2.935181  -1.890053  11.34
  -0.589243  25.599566  17.107493   4.372324  26.094118  19.130742
   4.438581  23.611003   6.960652  12.548072  21.62465   -6.255537
   6.813655  -3.207494   8.662606  18.629768   6.131886   8.892778
   3.721534  -4.252269  11.976238  15.4387    17.663482   9.992417
  24.940881  21.155161  21.259439  19.962318  -1.546576  12.36214
   5.578524   3.059346  -9.139197  14.507329  19.456031  -0.128112
   6.134812   4.834777  -3.701233  -4.421466   5.42848    9.186554
  20.750072   3.875541  11.549651  -1.129746  -1.547028  14.257146
  -1.597418   7.2618    19.838109  10.751806  11.880276  13.351948
   2.648789   2.354478  18.241645  19.260349  -8.564352  14.417385
 -21.829103  24.122843  12.478823   4.448164  18.35813   -5.981022
  21.427797  23.325154  16.463501  18.066585   7.739025  20.147325
  24.757052  26.731857  22.952908  19.794

### 5.4 Urban Heat Island Intensity

The `Map_2_5_intensity` layer contains point-based Urban Heat Island
(UHI) intensity information for European cities.

The spatial coordinates are stored in `Column1` and `Column2`,
while `Column3` represents UHI intensity in degrees Celsius.

The UHI intensity indicator is based on the spatial 90th percentile
(P90) UHI intensity for each city.

In [36]:
# Rename the heat intensity columns for clarity

heat_intensity = intensity.rename(
    columns={
        "Column1": "longitude",
        "Column2": "latitude",
        "Column3": "uhi_intensity_c"
    }
)

display(
    heat_intensity[
        ["longitude", "latitude", "uhi_intensity_c", "geometry"]
    ].head(10)
)

,longitude,latitude,uhi_intensity_c,geometry
0,-0.477337,38.362244,1.071040,POINT Z (3402258.625 1760840.363 1.071)
1,4.896728,52.364805,1.446170,POINT Z (3973698.674 3262772.191 1.446)
2,4.402000,51.246000,1.243220,POINT Z (3930553.314 3141075.188 1.243)
3,23.728056,37.983889,2.101180,POINT Z (5528466.128 1764761.749 2.101)
4,2.132641,41.384404,1.370220,POINT Z (3661587.218 2065827.861 1.37)
5,16.838012,41.122724,0.723279,POINT Z (4896838.966 2028577.647 0.723)
6,7.593117,47.561912,1.461090,POINT Z (4139786.084 2719423.963 1.461)
7,20.402604,44.813520,1.481330,POINT Z (5142343.508 2468663.147 1.481)
8,13.410963,52.518918,1.730470,POINT Z (4552446.382 3273166.94 1.731)
9,-2.935181,43.290576,0.953347,POINT Z (3274279.494 2332983.716 0.953)


### 5.5 Assign Heat Intensity to Cities

The UHI intensity points are spatially matched with the selected
Functional Urban Areas to identify the corresponding city.

This produces one city-level UHI intensity value for the selected
cities.

In [37]:
# Match UHI intensity points to the Urban Atlas city polygons

heat_city = gpd.sjoin(
    heat_intensity[
        ["uhi_intensity_c", "geometry"]
    ],
    urban_atlas[
        ["city", "geometry"]
    ],
    how="left",
    predicate="within"
)

# Keep one record per city

heat_city_indicators = (
    heat_city[
        ["city", "uhi_intensity_c"]
    ]
    .dropna(subset=["city"])
    .drop_duplicates(subset=["city"])
)

display(
    heat_city_indicators.sort_values("city")
)

,city,uhi_intensity_c
1,Amsterdam,1.44617
3,Athina,2.10118
4,Barcelona,1.37022
8,Berlin,1.73047
17,Budapest,1.99352
50,Madrid,1.72486
53,Milano,2.00832
71,Praha,1.70673
94,Wien,1.65995


### 5.6 Validate Heat–City Matching

The spatial join successfully matched the UHI intensity points to
nine of the ten selected cities.

Lisboa was not matched and therefore requires additional validation
before the final city-level heat indicator is created.

In [38]:
# Check which cities are missing from the UHI intensity match

selected_cities = set(
    urban_atlas["city"].str.upper().unique()
)

matched_cities = set(
    heat_city_indicators["city"].str.upper().unique()
)

missing_cities = sorted(selected_cities - matched_cities)

print("Missing cities:", missing_cities)

Missing cities: ['LISBOA']


### 5.6 Validate Lisboa Heat–City Matching

The spatial join matched nine of the ten selected cities.

Lisboa was not matched, so its heat-intensity point is inspected
directly using the longitude and latitude attributes provided
in the EEA heat layer.

In [39]:
# Inspect heat-intensity points around Lisboa

lisboa_candidates = heat_intensity[
    heat_intensity["longitude"].between(-10, -8) &
    heat_intensity["latitude"].between(38, 39.5)
][
    ["longitude", "latitude", "uhi_intensity_c", "geometry"]
]

display(lisboa_candidates)

,longitude,latitude,uhi_intensity_c,geometry
44,-9.139197,38.695177,1.11103,POINT Z (2664852.639 1943834.996 1.111)


### 5.7 Final UHI Intensity Indicator

The UHI intensity layer contains one representative intensity value
for each European city.

Lisboa was not captured by the polygon-based spatial join, so its
official heat point was validated directly using its longitude and
latitude and added to the city-level indicator.

In [40]:
# Add the validated Lisboa heat-intensity value

lisboa_heat = pd.DataFrame({
    "city": ["Lisboa"],
    "uhi_intensity_c": [1.11103]
})

# Standardize city names
heat_city_indicators["city"] = (
    heat_city_indicators["city"].str.title()
)

# Add Lisboa
heat_city_indicators = pd.concat(
    [heat_city_indicators, lisboa_heat],
    ignore_index=True
)

# Remove duplicates and sort
heat_city_indicators = (
    heat_city_indicators
    .drop_duplicates(subset="city")
    .sort_values("city")
    .reset_index(drop=True)
)

display(heat_city_indicators)

,city,uhi_intensity_c
0,Amsterdam,1.44617
1,Athina,2.10118
2,Barcelona,1.37022
3,Berlin,1.73047
4,Budapest,1.99352
5,Lisboa,1.11103
6,Madrid,1.72486
7,Milano,2.00832
8,Praha,1.70673
9,Wien,1.65995


### 5.8 Heatwave Indicator

The `Map_2_5_heatwaves` layer provides a heatwave-related indicator
for the selected European cities.

The variable `hw85_20_52` is inspected and matched to the selected
cities before being incorporated into the final city-level dataset.

In [41]:
# Inspect heatwave values

heatwaves_selected = heatwaves[
    ["Id", "hw85_20_52", "geometry"]
].copy()

print("Heatwave value range:")
print(heatwaves_selected["hw85_20_52"].describe())

display(
    heatwaves_selected.head(10)
)

Heatwave value range:
count    154.000000
mean       2.444805
std        2.130974
min        0.000000
25%        1.000000
50%        1.500000
75%        3.875000
max        9.500000
Name: hw85_20_52, dtype: float64


,Id,hw85_20_52,geometry
0,1,5.0,"POLYGON ((5692093.438 6008708.517, 5645357.918..."
1,2,1.5,"POLYGON ((5737530.076 6047571.912, 5692093.438..."
2,3,1.5,"POLYGON ((5645357.918 5971175.419, 5692093.438..."
3,4,0.0,"POLYGON ((4575723.655 4923741.954, 4556303.062..."
4,5,0.0,"POLYGON ((6173889.73 5578598.116, 5988898.265 ..."
5,7,0.0,"POLYGON ((5675265.501 5006428.332, 5518086.105..."
6,8,0.0,"POLYGON ((4613685.465 4512248.546, 4520904.752..."
7,9,1.5,"POLYGON ((4706220.493 4521273.482, 4613685.465..."
8,10,1.5,"POLYGON ((4890241.922 4546705.825, 4798431.924..."
9,12,0.0,"POLYGON ((4533560.601 4299068.377, 4434726.814..."


### 5.9 Heatwave Spatial Matching

The heatwave layer consists of spatial polygons representing
projected heatwave conditions.

The heatwave polygons are first matched to the spatial extent
of each selected Functional Urban Area to identify the relevant
heatwave features for each city.

In [42]:
# Identify heatwave polygons that spatially overlap each city extent

heatwave_candidates = []

for city in sorted(urban_atlas["city"].unique()):

    city_gdf = urban_atlas.loc[
        urban_atlas["city"] == city,
        ["geometry"]
    ]

    minx, miny, maxx, maxy = city_gdf.total_bounds

    candidates = heatwaves[
        (heatwaves.geometry.bounds["maxx"] >= minx) &
        (heatwaves.geometry.bounds["minx"] <= maxx) &
        (heatwaves.geometry.bounds["maxy"] >= miny) &
        (heatwaves.geometry.bounds["miny"] <= maxy)
    ].copy()

    candidates["city"] = city

    heatwave_candidates.append(candidates)

heatwave_candidates = pd.concat(
    heatwave_candidates,
    ignore_index=True
)

print("Candidate heatwave matches:", len(heatwave_candidates))

display(
    heatwave_candidates[
        ["city", "Id", "hw85_20_52"]
    ].drop_duplicates()
)

Candidate heatwave matches: 17


,city,Id,hw85_20_52
0,Amsterdam,62,0.0
1,Barcelona,104,4.5
2,Berlin,43,1.5
3,Berlin,62,0.0
4,Berlin,64,1.0
5,Budapest,64,1.0
6,Budapest,65,1.5
7,Budapest,93,2.5
8,Lisboa,121,1.5
9,Madrid,121,1.5


### 5.10 Heatwave Coverage Validation

The number of heatwave polygons associated with each city is
checked before calculating the city-level heatwave indicator.

This validation helps identify cities that are not directly
represented by a heatwave polygon within their spatial extent.

In [43]:
# Check the number of heatwave polygons associated with each city

heatwave_coverage = (
    heatwave_candidates
    .groupby("city")["Id"]
    .nunique()
    .reset_index(name="heatwave_polygons")
)

# Include cities with no matching heatwave polygon
all_cities = pd.DataFrame({
    "city": sorted(urban_atlas["city"].unique())
})

heatwave_coverage = all_cities.merge(
    heatwave_coverage,
    on="city",
    how="left"
)

heatwave_coverage["heatwave_polygons"] = (
    heatwave_coverage["heatwave_polygons"]
    .fillna(0)
    .astype(int)
)

display(heatwave_coverage)

,city,heatwave_polygons
0,Amsterdam,1
1,Athina,0
2,Barcelona,1
3,Berlin,3
4,Budapest,3
5,Lisboa,1
6,Madrid,2
7,Milano,1
8,Praha,2
9,Wien,3


### 5.11 Heatwave Indicator by City

Some cities overlap with more than one heatwave polygon.
To obtain a single city-level indicator, the mean value of
`hw85_20_52` across the relevant heatwave polygons is calculated.

The resulting indicator represents the average projected heatwave
value associated with the city's spatial extent.

In [44]:
# Calculate the mean heatwave indicator for each city

heatwave_city_indicators = (
    heatwave_candidates
    .groupby("city")["hw85_20_52"]
    .mean()
    .reset_index()
    .rename(columns={
        "hw85_20_52": "heatwave_indicator"
    })
)

display(
    heatwave_city_indicators
    .sort_values("heatwave_indicator", ascending=False)
)

,city,heatwave_indicator
1,Barcelona,4.500000
5,Madrid,2.000000
3,Budapest,1.666667
4,Lisboa,1.500000
6,Milano,1.500000
2,Berlin,0.833333
8,Wien,0.833333
7,Praha,0.500000
0,Amsterdam,0.000000


### 5.12 Validate Missing Heatwave Cities

Cities without a directly overlapping heatwave polygon are
identified separately.

Missing values are not replaced automatically because assigning
a distant heatwave polygon could introduce an unsupported
assumption into the analysis.

In [45]:
# Identify cities without a heatwave indicator

selected_cities = set(
    urban_atlas["city"].unique()
)

matched_heatwave_cities = set(
    heatwave_city_indicators["city"].unique()
)

missing_heatwave_cities = sorted(
    selected_cities - matched_heatwave_cities
)

print("Cities without a direct heatwave match:")
print(missing_heatwave_cities)

Cities without a direct heatwave match:
['Athina']


### 5.13 Inspect Missing Heatwave Cities

For cities without a direct heatwave match, the nearest heatwave
polygon is inspected only to understand the spatial relationship.

The nearest value is not automatically used as the city indicator.

In [46]:
# Recreate heat_city_points from the existing heat_city GeoDataFrame

heat_city_points = heat_city[
    ["city", "geometry"]
].copy()

display(heat_city_points.head())

,city,geometry
0,NaN,POINT Z (3402258.625 1760840.363 1.071)
1,Amsterdam,POINT Z (3973698.674 3262772.191 1.446)
2,NaN,POINT Z (3930553.314 3141075.188 1.243)
3,Athina,POINT Z (5528466.128 1764761.749 2.101)
4,Barcelona,POINT Z (3661587.218 2065827.861 1.37)


In [47]:
# Inspect the nearest heatwave polygon for missing cities

missing_city_points = heat_city_points[
    heat_city_points["city"].isin(missing_heatwave_cities)
].copy()

nearest_missing = gpd.sjoin_nearest(
    missing_city_points,
    heatwaves[
        ["Id", "hw85_20_52", "geometry"]
    ],
    how="left",
    distance_col="distance_m"
)

display(
    nearest_missing[
        ["city", "hw85_20_52", "distance_m"]
    ]
)

,city,hw85_20_52,distance_m
3,Athina,5.5,34718.228111


### 5.14 Final Heatwave Indicator

The final city-level heatwave indicator is based only on heatwave
polygons directly associated with each city's spatial extent.

Cities without a direct spatial match remain missing and are not
assigned values from distant polygons.

In [48]:
# Create the final heatwave indicator table

heatwave_city_indicators = (
    heatwave_city_indicators[
        ["city", "heatwave_indicator"]
    ]
    .copy()
)

# Add all selected cities
heatwave_city_indicators = (
    all_cities
    .merge(
        heatwave_city_indicators,
        on="city",
        how="left"
    )
)

display(
    heatwave_city_indicators
    .sort_values("city")
)

,city,heatwave_indicator
0,Amsterdam,0.000000
1,Athina,NaN
2,Barcelona,4.500000
3,Berlin,0.833333
4,Budapest,1.666667
5,Lisboa,1.500000
6,Madrid,2.000000
7,Milano,1.500000
8,Praha,0.500000
9,Wien,0.833333


### 5.15 Heat Indicators Summary

The heat analysis provides two complementary city-level indicators:

- `uhi_intensity_c`: Urban Heat Island intensity in degrees Celsius.
- `heatwave_indicator`: Average projected heatwave value from the
  heatwave polygons spatially associated with the city.

These indicators will be combined with green-space, street-tree,
and population indicators in the final city-level analysis.

In [49]:
# Combine UHI intensity and heatwave indicators

heat_indicators = (
    heat_city_indicators
    .merge(
        heatwave_city_indicators,
        on="city",
        how="left"
    )
    .sort_values("city")
    .reset_index(drop=True)
)

display(heat_indicators)

,city,uhi_intensity_c,heatwave_indicator
0,Amsterdam,1.44617,0.000000
1,Athina,2.10118,NaN
2,Barcelona,1.37022,4.500000
3,Berlin,1.73047,0.833333
4,Budapest,1.99352,1.666667
5,Lisboa,1.11103,1.500000
6,Madrid,1.72486,2.000000
7,Milano,2.00832,1.500000
8,Praha,1.70673,0.500000
9,Wien,1.65995,0.833333


### 5.16 Combine All City-Level Indicators

The environmental, demographic, and heat indicators are combined
into a single city-level dataset.

Missing heatwave values are retained as missing rather than replaced
with values from distant spatial features.

In [50]:
# Standardize city names for merging

city_indicators["city"] = (
    city_indicators["city"]
    .str.upper()
)

heat_indicators["city"] = (
    heat_indicators["city"]
    .str.upper()
)

# Merge heat indicators with the existing city-level indicators

final_city_indicators = city_indicators.merge(
    heat_indicators,
    on="city",
    how="left"
)

display(
    final_city_indicators.sort_values("city")
)

,city,green_space_pct,green_space_km2,street_tree_pct,population_2021,uhi_intensity_c,heatwave_indicator
9,AMSTERDAM,8.189335,272.476885,4.219192,2682860,1.44617,0.000000
1,ATHINA,47.222665,927.601791,3.254409,3418170,2.10118,NaN
0,BARCELONA,48.296473,1277.947018,3.817476,4767556,1.37022,4.500000
2,BERLIN,40.156175,3639.715336,5.840580,4595309,1.73047,0.833333
6,BUDAPEST,24.274084,1553.230263,4.261056,2842187,1.99352,1.666667
4,LISBOA,36.514981,1604.253378,1.846103,2743699,1.11103,1.500000
3,MADRID,37.972160,2990.688215,2.091410,6507965,1.72486,2.000000
8,MILANO,9.927272,309.229584,4.814992,4609706,2.00832,1.500000
5,PRAHA,28.370472,1635.195480,3.407796,2088432,1.70673,0.500000
7,WIEN,23.588846,2269.756597,1.929496,2840721,1.65995,0.833333


### 5.17 City-Level Master Dataset

The final city-level dataset combines environmental, demographic,
and heat-related indicators for the ten selected European cities.

This dataset serves as the main analytical table for the subsequent
comparison, visualization, and statistical analysis.

In [51]:
# Save the city-level master dataset

output_path = BASE_PATH / "data_processed"

output_path.mkdir(exist_ok=True)

final_city_indicators.to_csv(
    output_path / "city_level_indicators.csv",
    index=False
)

print(
    f"Saved to: {output_path / 'city_level_indicators.csv'}"
)

Saved to: /Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_processed/city_level_indicators.csv


In [52]:
# Final dataset check

print("Shape:", final_city_indicators.shape)

print("\nMissing values:")
print(final_city_indicators.isna().sum())

Shape: (10, 7)

Missing values:
city                  0
green_space_pct       0
green_space_km2       0
street_tree_pct       0
population_2021       0
uhi_intensity_c       0
heatwave_indicator    1
dtype: int64


## 6. Data Quality Check

Before comparing the selected cities, the final dataset is checked
for missing values, duplicated cities, and plausible indicator ranges.

In [53]:
# Check for duplicated cities

print("Duplicated cities:")
print(final_city_indicators["city"].duplicated().sum())

# Check indicator ranges

print("\nIndicator ranges:")

print(
    "Green space %:",
    final_city_indicators["green_space_pct"].min(),
    "to",
    final_city_indicators["green_space_pct"].max()
)

print(
    "Street tree %:",
    final_city_indicators["street_tree_pct"].min(),
    "to",
    final_city_indicators["street_tree_pct"].max()
)

print(
    "UHI intensity (°C):",
    final_city_indicators["uhi_intensity_c"].min(),
    "to",
    final_city_indicators["uhi_intensity_c"].max()
)

print(
    "Population:",
    final_city_indicators["population_2021"].min(),
    "to",
    final_city_indicators["population_2021"].max()
)

Duplicated cities:
0

Indicator ranges:
Green space %: 8.189334984607827 to 48.29647349890187
Street tree %: 1.8461029608492452 to 5.840579547280518
UHI intensity (°C): 1.11103 to 2.10118
Population: 2088432 to 6507965


### 7. Building 

### 7.1 Building Data Setup

The building analysis uses the GHS-OBAT R2024A dataset from the European Commission Joint Research Centre (JRC).

The raw building data is stored in the `data_raw/Building/` folder.

At this stage, we define the project and building data paths and verify that the building directory is available.

In [54]:
from pathlib import Path

# Define project and data paths
PROJECT_DIR = Path.cwd().parent

DATA_RAW = PROJECT_DIR / "data_raw"
BUILDING_DIR = DATA_RAW / "Building"

print("Project directory:")
print(PROJECT_DIR)

print("\nBuilding data directory:")
print(BUILDING_DIR)

print("\nBuilding directory exists:", BUILDING_DIR.exists())

Project directory:
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures

Building data directory:
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/Building

Building directory exists: True


### 7.2 Create the Building Raw Data Folder

The GHS-OBAT country-level building datasets are raw data and are stored separately in the `data_raw/Building/` folder.

Creating a dedicated folder keeps the raw building data organized and preserves the original files without modification.

In [55]:
# Create the Building raw data folder if it does not exist

BUILDING_DIR.mkdir(parents=True, exist_ok=True)

print("Building folder created:")
print(BUILDING_DIR)

print("\nFolder exists:", BUILDING_DIR.exists())

Building folder created:
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/Building

Folder exists: True


### 7.3 Check Building Folder Contents

The GHS-OBAT building datasets are stored in the `data_raw/Building/` folder.

This step verifies that the country-level building datasets are available in the correct raw-data location before starting the analysis.

In [56]:
# Check the contents of the Building folder

print("Building folder contents:\n")

for item in sorted(BUILDING_DIR.iterdir()):
    print("-", item.name)

Building folder contents:

- .DS_Store
- GHS_OBAT_CSV_AUT_E2020_R2024A_V1_0
- GHS_OBAT_CSV_CZE_E2020_R2024A_V1_0
- GHS_OBAT_CSV_DEU_E2020_R2024A_V1_0
- GHS_OBAT_CSV_ESP_E2020_R2024A_V1_0
- GHS_OBAT_CSV_GRC_E2020_R2024A_V1_0
- GHS_OBAT_CSV_HUN_E2020_R2024A_V1_0
- GHS_OBAT_CSV_ITA_E2020_R2024A_V1_0
- GHS_OBAT_CSV_NLD_E2020_R2024A_V1_0
- GHS_OBAT_CSV_PRT_E2020_R2024A_V1_0


In [57]:
# Check the country folders inside the Building directory

print("Building country folders:\n")

for item in sorted(BUILDING_DIR.iterdir()):
    if item.is_dir():
        print("📁", item.name)

Building country folders:

📁 GHS_OBAT_CSV_AUT_E2020_R2024A_V1_0
📁 GHS_OBAT_CSV_CZE_E2020_R2024A_V1_0
📁 GHS_OBAT_CSV_DEU_E2020_R2024A_V1_0
📁 GHS_OBAT_CSV_ESP_E2020_R2024A_V1_0
📁 GHS_OBAT_CSV_GRC_E2020_R2024A_V1_0
📁 GHS_OBAT_CSV_HUN_E2020_R2024A_V1_0
📁 GHS_OBAT_CSV_ITA_E2020_R2024A_V1_0
📁 GHS_OBAT_CSV_NLD_E2020_R2024A_V1_0
📁 GHS_OBAT_CSV_PRT_E2020_R2024A_V1_0


### 7.4 Locate the Germany (DEU) Building Data

The German GHS-OBAT dataset is used as the first test case because Berlin is one of the cities included in the project.

Before processing all countries, we locate the German CSV file and verify its path.

In [58]:
# Locate the German GHS-OBAT CSV file

DEU_DIR = BUILDING_DIR / "GHS_OBAT_CSV_DEU_E2020_R2024A_V1_0"

DEU_FILES = list(DEU_DIR.glob("*.csv"))

print("German CSV files found:", len(DEU_FILES))

for file in DEU_FILES:
    print(file)

German CSV files found: 1
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/Building/GHS_OBAT_CSV_DEU_E2020_R2024A_V1_0/GHS_OBAT_CSV_DEU_E2020_R2024A_V1_0.csv


### 7.5 Inspect the German Building Dataset

Before processing the full German building dataset, we inspect a small sample of rows and the available columns.

This allows us to understand the structure of the GHS-OBAT data and identify the variables needed to calculate city-level building indicators.

In [59]:
import pandas as pd

# Path to the German building CSV
DEU_CSV = DEU_FILES[0]

# Read only a small sample
deu_sample = pd.read_csv(
    DEU_CSV,
    nrows=5
)

print("Number of columns:", len(deu_sample.columns))

print("\nColumns:")
print(deu_sample.columns.tolist())

print("\nSample data:")
display(deu_sample)

Number of columns: 11

Columns:
['id', 'lon', 'lat', 'country', 'adm1', 'height', 'shapefactor', 'use', 'epoch', 'area', 'perimeter']

Sample data:


,id,lon,lat,country,adm1,height,shapefactor,use,epoch,area,perimeter
0,08b1f33971649fff0200c9bcf796088b,8.34980,54.62339,DEU,Schleswig-Holstein,NaN,NaN,0,0,5.70,9.60
1,08b1f33971203fff0200e051b389c2ab,8.35173,54.62808,DEU,Schleswig-Holstein,2.5,2.07558,1,2,9.92,12.66
2,08b1f33971202fff0200246e0e197590,8.35092,54.62819,DEU,Schleswig-Holstein,NaN,NaN,0,0,22.23,19.50
3,08b1f33971230fff020032e6fa98dc3c,8.34972,54.62884,DEU,Schleswig-Holstein,NaN,NaN,0,0,38.64,26.06
4,08b1f33971200fff0200045d4ec83521,8.35142,54.62874,DEU,Schleswig-Holstein,2.5,1.90076,1,1,13.51,14.87


### 7.6 Check Building Data Quality

Before calculating city-level building indicators, we perform basic quality checks on the German GHS-OBAT dataset.

We check the total number of building records, missing values, and duplicated building IDs using a chunk-based approach to avoid loading the entire dataset into memory.

In [60]:
# Basic quality check using chunks

total_rows = 0
missing_values = None
duplicate_ids = 0
seen_ids = set()

for chunk in pd.read_csv(
    DEU_CSV,
    chunksize=100_000
):

    total_rows += len(chunk)

    # Check missing values
    chunk_missing = chunk.isna().sum()

    if missing_values is None:
        missing_values = chunk_missing
    else:
        missing_values += chunk_missing

    # Check duplicate building IDs
    current_ids = set(chunk["id"].dropna())

    duplicate_ids += len(current_ids.intersection(seen_ids))
    seen_ids.update(current_ids)

print("Total building records:", total_rows)

print("\nMissing values:")
print(missing_values)

print("\nDuplicate building IDs:", duplicate_ids)

Total building records: 44109854

Missing values:
id                  0
lon                 0
lat                 0
country             0
adm1                0
height         488005
shapefactor    488005
use                 0
epoch               0
area                0
perimeter           0
dtype: int64

Duplicate building IDs: 0


### 7.7 Berlin Building Count — Approximate Bounding Box

A preliminary building count is calculated using an approximate geographic bounding box around Berlin.

This step is used only as an initial check and does not represent the final city-level building count because the bounding box may include areas outside the actual city boundary.

In [61]:
# Berlin approximate bounding box
# This is only a preliminary check.

BERLIN_MIN_LON = 13.05
BERLIN_MAX_LON = 13.75
BERLIN_MIN_LAT = 52.30
BERLIN_MAX_LAT = 52.70

berlin_building_count = 0

for chunk in pd.read_csv(
    DEU_CSV,
    usecols=["id", "lon", "lat"],
    chunksize=100_000
):
    
    berlin_mask = (
        (chunk["lon"] >= BERLIN_MIN_LON) &
        (chunk["lon"] <= BERLIN_MAX_LON) &
        (chunk["lat"] >= BERLIN_MIN_LAT) &
        (chunk["lat"] <= BERLIN_MAX_LAT)
    )
    
    berlin_building_count += berlin_mask.sum()

print(
    "Buildings within Berlin bounding box:",
    berlin_building_count
)

Buildings within Berlin bounding box: 988755


### 7.8 Use Berlin City Boundary

The previous step used a geographic bounding box to identify buildings around Berlin.

A bounding box can include areas outside the actual city boundary. Therefore, the next step checks the existing spatial datasets in the project to identify a suitable city boundary for a more accurate building analysis.

In [62]:
# Check the city-related spatial datasets already available in the project

print("Available variables related to city boundaries:")

for name in [
    "heat_city",
    "heat_city_indicators",
    "final_city_indicators"
]:
    if name in globals():
        print("-", name)

Available variables related to city boundaries:
- heat_city
- heat_city_indicators
- final_city_indicators


### 7.9 Inspect Berlin City Geometry

The available city-related GeoDataFrame is inspected to determine whether it contains city boundary geometries.

The geometry type and coordinate reference system (CRS) are checked before using the dataset for spatial filtering.

In [63]:
# Inspect the available city GeoDataFrame

print("Type:", type(heat_city))

print("\nShape:")
print(heat_city.shape)

print("\nColumns:")
print(heat_city.columns.tolist())

print("\nCRS:")
print(heat_city.crs)

print("\nGeometry types:")
print(heat_city.geometry.geom_type.value_counts())

print("\nFirst rows:")
display(heat_city.head())

Type: <class 'geopandas.geodataframe.GeoDataFrame'>

Shape:
(100, 4)

Columns:
['uhi_intensity_c', 'geometry', 'index_right', 'city']

CRS:
EPSG:3035

Geometry types:
Point    100
Name: count, dtype: int64

First rows:


,uhi_intensity_c,geometry,index_right,city
0,1.07104,POINT Z (3402258.625 1760840.363 1.071),NaN,NaN
1,1.44617,POINT Z (3973698.674 3262772.191 1.446),682419.0,Amsterdam
2,1.24322,POINT Z (3930553.314 3141075.188 1.243),NaN,NaN
3,2.10118,POINT Z (5528466.128 1764761.749 2.101),317200.0,Athina
4,1.37022,POINT Z (3661587.218 2065827.861 1.37),438337.0,Barcelona


### 7.10 Check Available GeoDataFrames

The notebook is checked for existing GeoDataFrames that may contain suitable city boundary geometries.

This helps determine whether an existing project dataset can be reused instead of introducing an additional external boundary dataset.

In [64]:
# Find GeoDataFrames currently available in the notebook

import geopandas as gpd

print("Available GeoDataFrames:\n")

for name, value in list(globals().items()):
    if isinstance(value, gpd.GeoDataFrame):
        print(f"- {name}: {value.shape}, CRS={value.crs}")

Available GeoDataFrames:

- sample_gdf: (90323, 11), CRS=EPSG:3035
- gdf: (7432, 2), CRS=EPSG:3035
- urban_atlas: (790454, 13), CRS=EPSG:3035
- sample_street_trees: (25375, 7), CRS=EPSG:3035
- street_trees: (532898, 8), CRS=EPSG:3035
- city_gdf: (90323, 1), CRS=EPSG:3035
- heatwaves: (154, 6), CRS=EPSG:3035
- intensity: (100, 6), CRS=EPSG:3035
- heat_intensity: (100, 6), CRS=EPSG:3035
- heat_city: (100, 4), CRS=EPSG:3035
- lisboa_candidates: (1, 4), CRS=EPSG:3035
- heatwaves_selected: (154, 3), CRS=EPSG:3035
- heatwave_candidates: (17, 7), CRS=EPSG:3035
- candidates: (3, 7), CRS=EPSG:3035
- heat_city_points: (100, 2), CRS=EPSG:3035
- missing_city_points: (1, 2), CRS=EPSG:3035
- nearest_missing: (1, 6), CRS=EPSG:3035


### 7.11 Inspect Urban Atlas

The Urban Atlas dataset is already available in the project and contains detailed urban land-use information.

Before using it for the building analysis, we inspect its columns, geometry types, and coordinate reference system to determine whether it can support the spatial identification of the selected cities.

In [65]:
# Inspect the Urban Atlas dataset

print("Shape:")
print(urban_atlas.shape)

print("\nColumns:")
print(urban_atlas.columns.tolist())

print("\nCRS:")
print(urban_atlas.crs)

print("\nGeometry types:")
print(urban_atlas.geometry.geom_type.value_counts())

print("\nSample:")
display(urban_atlas.head())

Shape:
(790454, 13)

Columns:
['country', 'fua_name', 'fua_code', 'code_2021', 'class_2021', 'prod_date', 'identifier', 'perimeter', 'area', 'comment', 'geometry', 'city', 'green_space']

CRS:
EPSG:3035

Geometry types:
MultiPolygon    790454
Name: count, dtype: int64

Sample:


,country,fua_name,fua_code,code_2021,class_2021,prod_date,identifier,perimeter,area,comment,geometry,city,green_space
0,AT,Wien,AT001L3,21000,Arable land (annual crops),2025-09,70626-AT001L3,584.488251,22755.177704,None,"MULTIPOLYGON (((4849627.919 2757717.629, 48496...",Wien,False
1,AT,Wien,AT001L3,23000,Pastures,2025-09,84190-AT001L3,1805.337560,119901.961521,None,"MULTIPOLYGON (((4847684.504 2757187.15, 484768...",Wien,False
2,AT,Wien,AT001L3,23000,Pastures,2025-09,84176-AT001L3,414.713551,10866.489328,None,"MULTIPOLYGON (((4847385.417 2757561.293, 48473...",Wien,False
3,AT,Wien,AT001L3,23000,Pastures,2025-09,84186-AT001L3,780.743799,17642.653582,None,"MULTIPOLYGON (((4847588.642 2757102.584, 48475...",Wien,False
4,AT,Wien,AT001L3,23000,Pastures,2025-09,83923-AT001L3,543.393359,12376.342939,None,"MULTIPOLYGON (((4841878.454 2755912.078, 48418...",Wien,False


### 7.12 Check Cities Available in Urban Atlas

The city names available in the Urban Atlas dataset are checked to confirm that the ten selected European cities are represented consistently.

This verification supports the later matching of building data to the selected cities.

In [66]:
# Check cities available in Urban Atlas

urban_cities = (
    urban_atlas["city"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("Number of cities:", len(urban_cities))

print("\nCities:")
print(urban_cities)

Number of cities: 10

Cities:
['Amsterdam', 'Athina', 'Barcelona', 'Berlin', 'Budapest', 'Lisboa', 'Madrid', 'Milano', 'Praha', 'Wien']


### 7.13 Create Berlin Urban Boundary

Urban Atlas contains multiple land-use polygons for Berlin. These polygons are combined into a single urban boundary that can be used to identify buildings located within the city.

In [67]:
# Extract Berlin polygons from Urban Atlas

berlin_atlas = urban_atlas[
    urban_atlas["city"].str.upper() == "BERLIN"
].copy()

print("Berlin Urban Atlas polygons:", len(berlin_atlas))

print("CRS:", berlin_atlas.crs)

print(
    "Total area (km²):",
    berlin_atlas.geometry.area.sum() / 1_000_000
)

Berlin Urban Atlas polygons: 82024
CRS: EPSG:3035
Total area (km²): 9063.899536245357


### 7.14 Inspect Berlin Urban Atlas Attributes

The Berlin Urban Atlas polygons are inspected to verify their city, Functional Urban Area, and land-use attributes before using them as the spatial boundary for the building analysis.

In [68]:
# Inspect Berlin Urban Atlas attributes

display(
    berlin_atlas[
        ["city", "fua_name", "fua_code", "code_2021", "class_2021"]
    ].drop_duplicates().head(30)
)

print("\nUnique FUA names:")
print(
    berlin_atlas["fua_name"]
    .dropna()
    .unique()
)

print("\nUnique FUA codes:")
print(
    berlin_atlas["fua_code"]
    .dropna()
    .unique()
)

,city,fua_name,fua_code,code_2021,class_2021
162300,Berlin,Berlin,DE001L1,12100,"Industrial, commercial, public, military and p..."
162301,Berlin,Berlin,DE001L1,31000,Forests
162302,Berlin,Berlin,DE001L1,21000,Arable land (annual crops)
162303,Berlin,Berlin,DE001L1,40000,Wetlands
162304,Berlin,Berlin,DE001L1,14130,Green urban areas (Unknown access conditions)
162305,Berlin,Berlin,DE001L1,11230,Discontinuous low density urban fabric (S.L. :...
162310,Berlin,Berlin,DE001L1,11220,Discontinuous medium density urban fabric (S.L...
162312,Berlin,Berlin,DE001L1,14200,Sports and leisure facilities
162313,Berlin,Berlin,DE001L1,50000,Water
162324,Berlin,Berlin,DE001L1,11210,Discontinuous dense urban fabric (S.L. : 50% -...



Unique FUA names:
<ArrowStringArray>
['Berlin']
Length: 1, dtype: str

Unique FUA codes:
<ArrowStringArray>
['DE001L1']
Length: 1, dtype: str


### 7.15 Create Berlin Bounding Box

A bounding box is created from the Berlin Urban Atlas polygons.
This provides an efficient spatial filter before processing the large GHS-OBAT building dataset.


In [69]:
# Get the bounding box of Berlin Urban Atlas polygons
# This is much faster than union_all()

berlin_bounds = berlin_atlas.total_bounds

minx, miny, maxx, maxy = berlin_bounds

print("Berlin Urban Atlas bounding box:")
print("minx:", minx)
print("miny:", miny)
print("maxx:", maxx)
print("maxy:", maxy)

Berlin Urban Atlas bounding box:
minx: 4487400.408399999
miny: 3211307.003799999
maxx: 4609936.890699999
maxy: 3345192.6136000007


### 7.16 Locate Germany Building Data

The German GHS-OBAT building CSV is located in the raw Building data folder.


In [70]:
# ==========================================
# Locate Germany building data
# ==========================================

GERMANY_BUILDING_CSV = (
    PROJECT_DIR
    / "data_raw"
    / "Building"
    / "GHS_OBAT_CSV_DEU_E2020_R2024A_V1_0"
    / "GHS_OBAT_CSV_DEU_E2020_R2024A_V1_0.csv"
)

print("Building file exists:", GERMANY_BUILDING_CSV.exists())

Building file exists: True


### 7.17 Load Germany Building Data
The required GHS-OBAT building attributes are loaded from the German dataset

In [71]:
# ==========================================
# Load Germany building data
# ==========================================

building_cols = [
    "id",
    "lon",
    "lat",
    "height",
    "shapefactor",
    "use",
    "epoch",
    "area",
    "perimeter"
]

buildings_de = pd.read_csv(
    GERMANY_BUILDING_CSV,
    usecols=building_cols
)

print("Buildings loaded:", len(buildings_de))

Buildings loaded: 44109854


### 7.18 Filter Buildings Using Berlin Bounding Box

The German building points are first filtered using the Berlin Urban Atlas bounding box.
The bounding box is converted from EPSG:3035 to WGS84 because the building coordinates are stored as longitude and latitude.


In [72]:
# ==========================================
# Filter buildings using Berlin bounding box
# ==========================================

minx, miny, maxx, maxy = berlin_atlas.total_bounds

# Convert Berlin EPSG:3035 bounding box to WGS84

from pyproj import Transformer

transformer = Transformer.from_crs(
    "EPSG:3035",
    "EPSG:4326",
    always_xy=True
)

min_lon, min_lat = transformer.transform(minx, miny)
max_lon, max_lat = transformer.transform(maxx, maxy)

print("Berlin bounding box in WGS84:")
print("Longitude:", min_lon, "to", max_lon)
print("Latitude :", min_lat, "to", max_lat)

# Filter buildings

berlin_buildings_bbox = buildings_de[
    (buildings_de["lon"] >= min_lon) &
    (buildings_de["lon"] <= max_lon) &
    (buildings_de["lat"] >= min_lat) &
    (buildings_de["lat"] <= max_lat)
].copy()

print("\nBuildings inside Berlin bounding box:")
print(len(berlin_buildings_bbox))

Berlin bounding box in WGS84:
Longitude: 12.422715155003235 to 14.320068092087462
Latitude : 51.98689200052724 to 53.13766113405813

Buildings inside Berlin bounding box:
2164130


### 7.19 Convert Berlin Building Candidates to GeoDataFrame

The filtered building coordinates are converted into spatial points and transformed to the same CRS as the Urban Atlas data.

In [73]:
# ==========================================
# Convert Berlin candidate buildings
# to GeoDataFrame
# ==========================================

berlin_buildings = gpd.GeoDataFrame(
    berlin_buildings_bbox,
    geometry=gpd.points_from_xy(
        berlin_buildings_bbox["lon"],
        berlin_buildings_bbox["lat"]
    ),
    crs="EPSG:4326"
)

# Convert to Urban Atlas CRS

berlin_buildings = berlin_buildings.to_crs("EPSG:3035")

print("Candidate buildings:", len(berlin_buildings))
print("CRS:", berlin_buildings.crs)

Candidate buildings: 2164130
CRS: EPSG:3035


### 7.20 Identify Buildings Inside Berlin

Filter the candidate buildings to those located within the actual Berlin Urban Atlas area.

In [74]:
# Create a single Berlin boundary from the Urban Atlas polygons

berlin_boundary = berlin_atlas.geometry.union_all()

print("Berlin boundary created successfully.")
print("Boundary type:", berlin_boundary.geom_type)

Berlin boundary created successfully.
Boundary type: Polygon


### 7.21 Building Analysis for All 10 Cities

Identify buildings located within each city's Urban Atlas boundary and calculate city-level building indicators.

In [75]:
cities = [
    "Amsterdam",
    "Athina",
    "Barcelona",
    "Berlin",
    "Budapest",
    "Lisboa",
    "Madrid",
    "Milano",
    "Praha",
    "Wien"
]

print("Cities:", cities)
print("Number of cities:", len(cities))

Cities: ['Amsterdam', 'Athina', 'Barcelona', 'Berlin', 'Budapest', 'Lisboa', 'Madrid', 'Milano', 'Praha', 'Wien']
Number of cities: 10


### 7.22 Check Building and Urban Atlas Data

The main building dataset and Urban Atlas dataset are checked to confirm that the required data are available before starting the analysis for all ten cities.

In [76]:
# Check the main building dataset

print("Building dataset available:", "buildings_de" in globals())

if "buildings_de" in globals():
    print("Building records:", len(buildings_de))
    print("Building columns:", list(buildings_de.columns))

print("\nUrban Atlas available:", "urban_atlas" in globals())

Building dataset available: True
Building records: 44109854
Building columns: ['id', 'lon', 'lat', 'height', 'shapefactor', 'use', 'epoch', 'area', 'perimeter']

Urban Atlas available: True


### 7.23 Load Urban Atlas Data for the 10 Cities

Load the Urban Atlas boundaries for all selected cities from their existing ZIP files.

In [77]:
import geopandas as gpd
import zipfile
import tempfile
from pathlib import Path

URBAN_ATLAS_ZIP_DIR = (
    PROJECT_DIR
    / "data_raw"
    / "urban_atlas"
    / "ZIP"
)

urban_atlas_cities = {}

for city in cities:

    # Find the ZIP file for the city
    zip_files = list(
        URBAN_ATLAS_ZIP_DIR.glob(f"*_{city.upper()}_*.zip")
    )

    if len(zip_files) == 0:
        print(f"❌ No Urban Atlas ZIP found for {city}")
        continue

    zip_path = zip_files[0]

    # Find the FlatGeobuf file inside the ZIP
    with zipfile.ZipFile(zip_path, "r") as z:

        fgb_files = [
            f for f in z.namelist()
            if f.lower().endswith(".fgb")
        ]

        if len(fgb_files) == 0:
            print(f"❌ No FGB file found for {city}")
            continue

        fgb_file = fgb_files[0]

        # Temporary extraction
        temp_dir = Path(tempfile.mkdtemp())
        z.extract(fgb_file, temp_dir)

    fgb_path = temp_dir / fgb_file

    # Read Urban Atlas
    city_atlas = gpd.read_file(fgb_path)

    urban_atlas_cities[city] = city_atlas

    print(
        f"✓ {city}: "
        f"{len(city_atlas):,} polygons | "
        f"CRS={city_atlas.crs}"
    )

print("\nUrban Atlas datasets loaded:", len(urban_atlas_cities))

✓ Amsterdam: 55,021 polygons | CRS=EPSG:3035
✓ Athina: 87,254 polygons | CRS=EPSG:3035
✓ Barcelona: 70,964 polygons | CRS=EPSG:3035
✓ Berlin: 82,024 polygons | CRS=EPSG:3035
✓ Budapest: 55,629 polygons | CRS=EPSG:3035
✓ Lisboa: 94,164 polygons | CRS=EPSG:3035
✓ Madrid: 105,685 polygons | CRS=EPSG:3035
✓ Milano: 77,413 polygons | CRS=EPSG:3035
✓ Praha: 71,977 polygons | CRS=EPSG:3035
✓ Wien: 90,323 polygons | CRS=EPSG:3035

Urban Atlas datasets loaded: 10


### 7.24 Building Data by Country

The GHS-OBAT building data is organized by country.

Each selected city is mapped to its corresponding country dataset so that the appropriate building data can be used during the city-level analysis.

In [78]:
city_country = {
    "Amsterdam": "NLD",
    "Athina": "GRC",
    "Barcelona": "ESP",
    "Berlin": "DEU",
    "Budapest": "HUN",
    "Lisboa": "PRT",
    "Madrid": "ESP",
    "Milano": "ITA",
    "Praha": "CZE",
    "Wien": "AUT"
}

print(city_country)

{'Amsterdam': 'NLD', 'Athina': 'GRC', 'Barcelona': 'ESP', 'Berlin': 'DEU', 'Budapest': 'HUN', 'Lisboa': 'PRT', 'Madrid': 'ESP', 'Milano': 'ITA', 'Praha': 'CZE', 'Wien': 'AUT'}


## 7.25 Extract Building Candidates by Country

Read each country's GHS-OBAT CSV and keep only building records that fall within the bounding box of the selected city or cities.

In [79]:
from pathlib import Path
import pandas as pd

BUILDING_DIR = PROJECT_DIR / "data_raw" / "Building"

building_files = {}

for country in sorted(set(city_country.values())):

    matches = list(
        BUILDING_DIR.glob(
            f"GHS_OBAT_CSV_{country}_E2020_R2024A_V1_0/*.csv"
        )
    )

    if len(matches) == 0:
        print(f"❌ File not found: {country}")
    else:
        building_files[country] = matches[0]
        print(f"✓ {country}: {matches[0].name}")

print("\nBuilding country files found:", len(building_files))

✓ AUT: GHS_OBAT_CSV_AUT_E2020_R2024A_V1_0.csv
✓ CZE: GHS_OBAT_CSV_CZE_E2020_R2024A_V1_0.csv
✓ DEU: GHS_OBAT_CSV_DEU_E2020_R2024A_V1_0.csv
✓ ESP: GHS_OBAT_CSV_ESP_E2020_R2024A_V1_0.csv
✓ GRC: GHS_OBAT_CSV_GRC_E2020_R2024A_V1_0.csv
✓ HUN: GHS_OBAT_CSV_HUN_E2020_R2024A_V1_0.csv
✓ ITA: GHS_OBAT_CSV_ITA_E2020_R2024A_V1_0.csv
✓ NLD: GHS_OBAT_CSV_NLD_E2020_R2024A_V1_0.csv
✓ PRT: GHS_OBAT_CSV_PRT_E2020_R2024A_V1_0.csv

Building country files found: 9


## 7.26 Extract Building Candidates Efficiently

Read the GHS-OBAT files in chunks and keep only buildings within each selected city's bounding box.

In [80]:
# ==========================================
# Extract building candidates efficiently
# ==========================================

building_candidates = []

# Create bounding boxes for all cities
city_bounds = {}

for city in cities:

    atlas = urban_atlas_cities[city]

    minx, miny, maxx, maxy = atlas.total_bounds

    city_bounds[city] = {
        "minx": minx,
        "miny": miny,
        "maxx": maxx,
        "maxy": maxy
    }


# Process one country at a time
for country, file_path in building_files.items():

    country_cities = [
        city for city in cities
        if city_country[city] == country
    ]

    print(f"\nProcessing {country} → {country_cities}")

    # Read CSV in chunks
    for chunk in pd.read_csv(
        file_path,
        usecols=[
            "id",
            "lon",
            "lat",
            "height",
            "shapefactor",
            "use",
            "epoch",
            "area",
            "perimeter"
        ],
        chunksize=500_000
    ):

        for city in country_cities:

            bounds = city_bounds[city]

            min_lon, min_lat = transformer.transform(
                bounds["minx"],
                bounds["miny"]
            )

            max_lon, max_lat = transformer.transform(
                bounds["maxx"],
                bounds["maxy"]
            )

            selected = chunk[
                (chunk["lon"] >= min_lon) &
                (chunk["lon"] <= max_lon) &
                (chunk["lat"] >= min_lat) &
                (chunk["lat"] <= max_lat)
            ].copy()

            if len(selected) > 0:

                selected["city"] = city

                building_candidates.append(selected)

    print(f"✓ Finished {country}")


# Combine all candidates
building_candidates_df = pd.concat(
    building_candidates,
    ignore_index=True
)

print("\n====================================")
print("BUILDING CANDIDATE EXTRACTION DONE")
print("====================================")

print(
    "Total candidate buildings:",
    f"{len(building_candidates_df):,}"
)

print(
    "Cities represented:",
    building_candidates_df["city"].unique()
)

display(
    building_candidates_df.groupby("city")
    .size()
    .reset_index(name="building_candidates")
)


Processing AUT → ['Wien']
✓ Finished AUT

Processing CZE → ['Praha']
✓ Finished CZE

Processing DEU → ['Berlin']
✓ Finished DEU

Processing ESP → ['Barcelona', 'Madrid']
✓ Finished ESP

Processing GRC → ['Athina']
✓ Finished GRC

Processing HUN → ['Budapest']
✓ Finished HUN

Processing ITA → ['Milano']
✓ Finished ITA

Processing NLD → ['Amsterdam']
✓ Finished NLD

Processing PRT → ['Lisboa']
✓ Finished PRT

BUILDING CANDIDATE EXTRACTION DONE
Total candidate buildings: 12,180,748
Cities represented: <ArrowStringArray>
[     'Wien',     'Praha',    'Berlin', 'Barcelona',    'Madrid',    'Athina',
  'Budapest',    'Milano', 'Amsterdam',    'Lisboa']
Length: 10, dtype: str


,city,building_candidates
0,Amsterdam,2061390
1,Athina,737136
2,Barcelona,524042
3,Berlin,2164130
4,Budapest,1529373
5,Lisboa,951408
6,Madrid,621177
7,Milano,1105985
8,Praha,1279362
9,Wien,1206745


## 7.27 Spatial Join: Buildings Inside Urban Atlas Boundaries

Keep only building points that are actually located inside the Urban Atlas boundary of each city.

In [81]:
import geopandas as gpd

# Convert building candidates to GeoDataFrame
building_gdf = gpd.GeoDataFrame(
    building_candidates_df,
    geometry=gpd.points_from_xy(
        building_candidates_df["lon"],
        building_candidates_df["lat"]
    ),
    crs="EPSG:4326"
)

# Convert to the same CRS as Urban Atlas
building_gdf = building_gdf.to_crs("EPSG:3035")

print("Building candidates:", f"{len(building_gdf):,}")
print("CRS:", building_gdf.crs)

# Store final buildings inside Urban Atlas
buildings_inside_city = []

for city in cities:

    print(f"Processing: {city}")

    city_boundary = urban_atlas_cities[city]

    boundary = city_boundary[["geometry"]].copy()

    city_buildings = gpd.sjoin(
        building_gdf[building_gdf["city"] == city],
        boundary,
        predicate="within",
        how="inner"
    )

    if "index_right" in city_buildings.columns:
        city_buildings = city_buildings.drop(
            columns=["index_right"]
        )

    buildings_inside_city.append(city_buildings)

    print(
        f"✓ {city}: {len(city_buildings):,} buildings"
    )


# Combine all cities
buildings_inside_city = gpd.GeoDataFrame(
    pd.concat(
        buildings_inside_city,
        ignore_index=True
    ),
    crs="EPSG:3035"
)

print("\n======================================")
print("BUILDINGS INSIDE CITY BOUNDARIES")
print("======================================")

print(
    "Total buildings:",
    f"{len(buildings_inside_city):,}"
)

display(
    buildings_inside_city
    .groupby("city")
    .size()
    .reset_index(name="building_count")
)

Building candidates: 12,180,748
CRS: EPSG:3035
Processing: Amsterdam
✓ Amsterdam: 1,518,060 buildings
Processing: Athina
✓ Athina: 674,312 buildings
Processing: Barcelona
✓ Barcelona: 473,673 buildings
Processing: Berlin
✓ Berlin: 1,719,162 buildings
Processing: Budapest
✓ Budapest: 1,111,861 buildings
Processing: Lisboa
✓ Lisboa: 763,599 buildings
Processing: Madrid
✓ Madrid: 525,884 buildings
Processing: Milano
✓ Milano: 733,763 buildings
Processing: Praha
✓ Praha: 908,593 buildings
Processing: Wien
✓ Wien: 1,010,181 buildings

BUILDINGS INSIDE CITY BOUNDARIES
Total buildings: 9,439,088


,city,building_count
0,Amsterdam,1518060
1,Athina,674312
2,Barcelona,473673
3,Berlin,1719162
4,Budapest,1111861
5,Lisboa,763599
6,Madrid,525884
7,Milano,733763
8,Praha,908593
9,Wien,1010181


##  KPI 1 — Building Density

Building Density = Number of Buildings / City Area (km²)

City area is calculated from the Urban Atlas polygons.

In [82]:
# ==========================================
# KPI 1 — BUILDING DENSITY
# ==========================================

city_area_records = []

for city in cities:

    gdf = urban_atlas_cities[city]

    area_km2 = gdf.geometry.area.sum() / 1_000_000

    city_area_records.append({
        "city": city,
        "area_km2": area_km2
    })

# Create city area table
city_area = pd.DataFrame(city_area_records)

# Count buildings per city
building_counts = (
    buildings_inside_city
    .groupby("city")
    .size()
    .reset_index(name="building_count")
)

# Merge building counts with city area
building_density = building_counts.merge(
    city_area,
    on="city",
    how="left"
)

# Calculate building density
building_density["building_density_per_km2"] = (
    building_density["building_count"] /
    building_density["area_km2"]
)

# Sort from highest to lowest
building_density = building_density.sort_values(
    "building_density_per_km2",
    ascending=False
).reset_index(drop=True)

print("========== KPI 1: BUILDING DENSITY ==========\n")

display(
    building_density[
        [
            "city",
            "building_count",
            "area_km2",
            "building_density_per_km2"
        ]
    ]
)

========== KPI 1: BUILDING DENSITY ==========



,city,building_count,area_km2,building_density_per_km2
0,Amsterdam,1518060,3327.216261,456.255284
1,Athina,674312,1964.314802,343.281026
2,Milano,733763,3114.950192,235.561712
3,Berlin,1719162,9063.899536,189.671343
4,Barcelona,473673,2646.046234,179.011611
5,Lisboa,763599,4393.411512,173.805481
6,Budapest,1111861,6398.718350,173.763079
7,Praha,908593,5763.723247,157.639942
8,Wien,1010181,9622.160263,104.984845
9,Madrid,525884,7876.002266,66.770423


## KPI 2 — Average Building Height

Average Building Height represents the mean building height within each city's Urban Atlas area.

The calculation uses the available building height values from the GHS-OBAT dataset.

In [83]:
# ==========================================
# KPI 2 — AVERAGE BUILDING HEIGHT
# ==========================================

average_building_height = (
    buildings_inside_city
    .groupby("city")["height"]
    .mean()
    .reset_index(name="average_building_height_m")
)

# Sort from highest to lowest
average_building_height = average_building_height.sort_values(
    "average_building_height_m",
    ascending=False
).reset_index(drop=True)

print("========== KPI 2: AVERAGE BUILDING HEIGHT ==========\n")

display(
    average_building_height[
        [
            "city",
            "average_building_height_m"
        ]
    ]
)

========== KPI 2: AVERAGE BUILDING HEIGHT ==========



,city,average_building_height_m
0,Barcelona,10.610237
1,Athina,9.769938
2,Milano,9.383429
3,Madrid,8.078552
4,Amsterdam,7.356376
5,Lisboa,6.952854
6,Berlin,5.459626
7,Budapest,5.231145
8,Wien,5.114686
9,Praha,4.772399


## Building KPI Table

## Building KPI Table

Combine the completed building KPIs into one city-level table.

This table contains the building indicators calculated for the ten selected cities and will be merged with the other city-level indicators later.

In [84]:
# ==========================================
# BUILDING KPI TABLE
# ==========================================

building_kpi_table = building_density[
    ["city", "building_density_per_km2"]
].merge(
    average_building_height[
        ["city", "average_building_height_m"]
    ],
    on="city",
    how="outer"
)

print("Building KPI table:")
display(building_kpi_table)

Building KPI table:


,city,building_density_per_km2,average_building_height_m
0,Amsterdam,456.255284,7.356376
1,Athina,343.281026,9.769938
2,Barcelona,179.011611,10.610237
3,Berlin,189.671343,5.459626
4,Budapest,173.763079,5.231145
5,Lisboa,173.805481,6.952854
6,Madrid,66.770423,8.078552
7,Milano,235.561712,9.383429
8,Praha,157.639942,4.772399
9,Wien,104.984845,5.114686


## Add Building KPIs to Final City Indicators

Merge building density and average building height into the final city-level indicators.

In [85]:
# ==========================================
# ADD BUILDING KPIs TO FINAL CITY INDICATORS
# ==========================================

# Make city names consistent
building_kpi_merge = building_kpi_table.copy()

building_kpi_merge["city"] = (
    building_kpi_merge["city"]
    .str.upper()
    .str.strip()
)

final_city_indicators["city"] = (
    final_city_indicators["city"]
    .str.upper()
    .str.strip()
)

# Remove old Building KPI columns if they already exist
for col in [
    "building_density_per_km2",
    "average_building_height_m"
]:
    if col in final_city_indicators.columns:
        final_city_indicators = final_city_indicators.drop(
            columns=col
        )

# Add Building KPIs
final_city_indicators = final_city_indicators.merge(
    building_kpi_merge[
        [
            "city",
            "building_density_per_km2",
            "average_building_height_m"
        ]
    ],
    on="city",
    how="left"
)

print("Building KPIs added successfully.")

display(
    final_city_indicators[
        [
            "city",
            "building_density_per_km2",
            "average_building_height_m"
        ]
    ]
)

Building KPIs added successfully.


,city,building_density_per_km2,average_building_height_m
0,BARCELONA,179.011611,10.610237
1,ATHINA,343.281026,9.769938
2,BERLIN,189.671343,5.459626
3,MADRID,66.770423,8.078552
4,LISBOA,173.805481,6.952854
5,PRAHA,157.639942,4.772399
6,BUDAPEST,173.763079,5.231145
7,WIEN,104.984845,5.114686
8,MILANO,235.561712,9.383429
9,AMSTERDAM,456.255284,7.356376


# Statistical Evidence Stage

## Research hypotheses

The inferential analysis must be based on a common spatial unit. It must not duplicate a single city-level UHI value across many cells.

Primary hypotheses:

**H1, Green infrastructure and heat**

H0: Local green-space availability is not associated with local LST.

H1: Higher local green-space availability is associated with lower local LST.

**H2, Built surface and heat**

H0: Local impervious or built-surface intensity is not associated with local LST.

H1: Higher local impervious or built-surface intensity is associated with higher local LST.

Additional variables such as population density, street-tree coverage, and building density are tested only when they can be aligned to the same defensible spatial unit.

The analysis reports sample size, effect direction and size, p-values, confidence information where appropriate, and limitations. Statistical association is not described as causation.


In [86]:
# Statistical-analysis dependencies
import numpy as np
import pandas as pd
from scipy import stats

ALPHA = 0.05

print("Statistical evidence stage ready.")
print("Alpha:", ALPHA)


Statistical evidence stage ready.
Alpha: 0.05


## Spatial evidence validation

Before hypothesis testing, the notebook must verify that LST and explanatory variables are available at compatible spatial resolution.

The next analysis should create one common spatial analytical table with one row per valid spatial unit and columns for:

- city
- LST
- green-space share
- impervious or built-surface share
- population density, if spatially compatible
- street-tree coverage, if spatially compatible
- building density, if spatially compatible

No Priority Score or recommendation is calculated before this validation succeeds.


In [87]:
# Expected spatial evidence table schema.
# This cell is a validation gate, not fabricated data.

required_core = [
    "city",
    "lst",
    "green_space_share",
    "impervious_share"
]

if "spatial_evidence" in globals():
    missing = [c for c in required_core if c not in spatial_evidence.columns]
    if missing:
        raise ValueError(f"Spatial evidence table is missing: {missing}")
    print("Spatial evidence rows:", len(spatial_evidence))
    print("Cities:", spatial_evidence["city"].nunique())
    display(spatial_evidence[required_core].describe(include="all"))
else:
    print(
        "NEXT REQUIRED STEP: build `spatial_evidence` from the existing "
        "LST and spatial raw data before running inferential tests."
    )


NEXT REQUIRED STEP: build `spatial_evidence` from the existing LST and spatial raw data before running inferential tests.


## Spatial LST Data Inspection

Before building the spatial evidence table, the Land Surface Temperature (LST) dataset is inspected to identify its spatial dimensions, coordinates, and temperature variable.

This step ensures that the heat data can be correctly aligned with the urban spatial indicators.

In [88]:
import xarray as xr
from pathlib import Path

lst_path = Path(
    "/Users/nedalgubran/Desktop/Project/"
    "Smart_Cities_Cooler_Futures/data_raw/LST/LST_2024_August.nc"
)

print("File exists:", lst_path.exists())
print("File size:", lst_path.stat().st_size if lst_path.exists() else "Not found")

for engine in ["netcdf4", "h5netcdf", "scipy"]:
    try:
        lst_ds = xr.open_dataset(lst_path, engine=engine)
        print(f"\nOpened successfully with: {engine}")
        print(lst_ds)
        print("\nDATA VARIABLES:")
        print(list(lst_ds.data_vars))
        print("\nCOORDINATES:")
        print(list(lst_ds.coords))
        break
    except Exception as e:
        print(f"\n{engine} failed:")
        print(type(e).__name__, str(e)[:300])

File exists: True
File size: 39886152

netcdf4 failed:
OSError [Errno -51] NetCDF: Unknown file format: '/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/LST/LST_2024_August.nc'

h5netcdf failed:
OSError Unable to synchronously open file (file signature not found)

scipy failed:
TypeError Error: /Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/LST/LST_2024_August.nc is not a valid NetCDF 3 file
            If this is a NetCDF4 file, you may need to install the
            netcdf4 library, e.g.,

            $ pip install netcdf4
            


In [89]:
from pathlib import Path

lst_path = Path(
    "/Users/nedalgubran/Desktop/Project/"
    "Smart_Cities_Cooler_Futures/data_raw/LST/LST_2024_August.nc"
)

with open(lst_path, "rb") as f:
    header = f.read(200)

print("First 200 bytes:")
print(header)

First 200 bytes:
b'PK\x03\x04\x14\x00\x00\x00\x08\x00jo\']1\x95\xa2\x19&\x9c`\x02\\\x84`\x02`\x00\x00\x00ESACCI-LST-L3S-LST-IRCDR_-0.01deg_1MONTHLY_DAY-20240801000000-fv3.00.area-subset.53.25.37.-10.nc\x00\x00\x80\xff\x7f\x89HDF\r\n\x1a\n\x02\x08\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xff\xff\xff\xff\xff\xff\xff\\\x84`\x02\x00\x00\x00\x000\x00\x00\x00\x00\x00\x00\x00d\x1c\x99}OHDR\x02\r\xf1\x02\x02"\x00\x00\x00\x00\x00\x03\x17\x00\x00\x00\x00'


In [90]:
import zipfile
from pathlib import Path
import xarray as xr

zip_path = Path(
    "/Users/nedalgubran/Desktop/Project/"
    "Smart_Cities_Cooler_Futures/data_raw/LST/LST_2024_August.nc"
)

extract_dir = zip_path.parent / "extracted"
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside ZIP:")
    print(z.namelist())

    z.extractall(extract_dir)
    real_nc = extract_dir / z.namelist()[0]

print("\nExtracted file:")
print(real_nc)

lst_ds = xr.open_dataset(real_nc, engine="netcdf4")

print("\nDATASET:")
print(lst_ds)

print("\nDATA VARIABLES:")
print(list(lst_ds.data_vars))

print("\nCOORDINATES:")
print(list(lst_ds.coords))

Files inside ZIP:
['ESACCI-LST-L3S-LST-IRCDR_-0.01deg_1MONTHLY_DAY-20240801000000-fv3.00.area-subset.53.25.37.-10.nc']

Extracted file:
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/LST/extracted/ESACCI-LST-L3S-LST-IRCDR_-0.01deg_1MONTHLY_DAY-20240801000000-fv3.00.area-subset.53.25.37.-10.nc

DATASET:
<xarray.Dataset> Size: 291MB
Dimensions:          (time: 1, lat: 1600, lon: 3500, length_scale: 1, channel: 2)
Coordinates:
  * time             (time) datetime64[ns] 8B 2024-08-01
  * lat              (lat) float32 6kB 37.01 37.01 37.02 ... 52.98 52.98 52.99
  * lon              (lon) float32 14kB -9.995 -9.985 -9.975 ... 24.98 24.99
  * channel          (channel) float32 8B 11.0 12.0
Dimensions without coordinates: length_scale
Data variables: (12/14)
    dtime            (time, lat, lon) float32 22MB ...
    satze            (time, lat, lon) float32 22MB ...
    sataz            (time, lat, lon) float32 22MB ...
    solze            (time, lat, lon) float32 22

## LST Variable Validation

The LST variable is validated before spatial analysis to confirm its units, value range, missing-data structure, and spatial resolution.

In [91]:
lst = lst_ds["lst"]

print("LST attributes:")
print(lst.attrs)

print("\nShape:", lst.shape)
print("Data type:", lst.dtype)

print("\nLatitude resolution:")
print(float(lst_ds.lat[1] - lst_ds.lat[0]))

print("\nLongitude resolution:")
print(float(lst_ds.lon[1] - lst_ds.lon[0]))

print("\nLST summary:")
print(
    lst.isel(time=0)
       .to_series()
       .describe()
)

LST attributes:
{'long_name': 'land surface temperature', 'units': 'kelvin', 'valid_min': np.int16(-8315), 'valid_max': np.int16(7685), 'actual_range': array([190.10999, 343.4    ], dtype=float32)}

Shape: (1, 1600, 3500)
Data type: float32

Latitude resolution:
0.0099945068359375

Longitude resolution:
0.0099945068359375

LST summary:
count    3.571772e+06
mean     3.057186e+02
std      6.557942e+00
min      1.913300e+02
25%      3.008600e+02
50%      3.045800e+02
75%      3.100300e+02
max      3.333400e+02
Name: lst, dtype: float64


## Spatial LST Preparation

The LST data is converted from Kelvin to Celsius and spatially prepared for the ten study cities.

Rather than using one heat value per city, the analysis preserves local spatial variation within each city. This provides the basis for testing whether urban characteristics are associated with higher or lower surface temperature.

In [92]:
# Convert LST from Kelvin to Celsius
lst_c = lst_ds["lst"].isel(time=0) - 273.15
lst_c.name = "lst_c"

print("LST converted to Celsius.")
print("Valid observations:", int(lst_c.count()))

print("\nTemperature summary (°C):")
print(
    lst_c.to_series()
         .describe()
         .round(2)
)

LST converted to Celsius.
Valid observations: 3571772

Temperature summary (°C):
count    3571772.00
mean          32.57
std            6.56
min          -81.82
25%           27.71
50%           31.43
75%           36.88
max           60.19
Name: lst_c, dtype: float64


## LST Quality Control

Before spatial analysis, the LST observations are checked using the quality information provided with the dataset.

Invalid or low-quality observations are excluded according to the dataset metadata rather than by applying an arbitrary temperature threshold.

In [93]:
# Inspect quality-related variables and metadata

quality_vars = [
    var for var in lst_ds.data_vars
    if any(key in var.lower() for key in ["quality", "flag", "lcc", "n"])
]

print("Quality-related variables:")
print(quality_vars)

for var in quality_vars:
    print(f"\n--- {var} ---")
    print(lst_ds[var].attrs)

Quality-related variables:
['lst_uncertainty', 'lst_unc_ran', 'lst_unc_loc_atm', 'lst_unc_loc_sfc', 'lst_unc_sys', 'lcc', 'n', 'lst_unc_loc_cor']

--- lst_uncertainty ---
{'long_name': 'land surface temperature total uncertainty', 'units': 'kelvin', 'valid_min': np.int16(0), 'valid_max': np.int16(10000), 'actual_range': array([3.2610002, 9.990001 ], dtype=float32)}

--- lst_unc_ran ---
{'long_name': 'uncertainty from uncorrelated errors', 'units': 'kelvin', 'valid_min': np.int16(0), 'valid_max': np.int16(10000), 'actual_range': array([0.       , 6.3170004], dtype=float32)}

--- lst_unc_loc_atm ---
{'long_name': 'uncertainty from locally correlated errors on atmospheric scales', 'units': 'kelvin', 'valid_min': np.int16(0), 'valid_max': np.int16(10000), 'actual_range': array([0.017, 4.486], dtype=float32)}

--- lst_unc_loc_sfc ---
{'long_name': 'uncertainty from locally correlated errors on surface scales', 'units': 'kelvin', 'valid_min': np.int16(0), 'valid_max': np.int16(10000), 'actua

## LST Quality Assessment

Before using LST observations in the statistical analysis, data quality is assessed using the number of clear-sky observations and the reported LST uncertainty.

This prevents low-confidence temperature observations from being treated as equally reliable evidence.

In [94]:
quality_df = pd.DataFrame({
    "lst_c": lst_c.values.ravel(),
    "n_clear": lst_ds["n"].isel(time=0).values.ravel(),
    "lst_uncertainty": lst_ds["lst_uncertainty"].isel(time=0).values.ravel()
}).dropna()

print("Quality observations:", len(quality_df))

print("\nClear-sky observations:")
print(quality_df["n_clear"].describe())

print("\nLST uncertainty (K):")
print(quality_df["lst_uncertainty"].describe())

Quality observations: 3413440

Clear-sky observations:
count    3.413440e+06
mean     5.238619e+00
std      1.717882e+00
min      1.000000e+00
25%      4.000000e+00
50%      5.000000e+00
75%      6.000000e+00
max      1.000000e+01
Name: n_clear, dtype: float64

LST uncertainty (K):
count    3.413440e+06
mean     5.178243e+00
std      1.667925e-01
min      4.606000e+00
25%      5.062000e+00
50%      5.167000e+00
75%      5.275000e+00
max      6.630000e+00
Name: lst_uncertainty, dtype: float64


In [95]:
for name, obj in list(globals().items()):
    if isinstance(obj, gpd.GeoDataFrame):
        print(
            name,
            "| rows:", len(obj),
            "| CRS:", obj.crs,
            "| columns:", list(obj.columns)[:10]
        )

sample_gdf | rows: 90323 | CRS: EPSG:3035 | columns: ['country', 'fua_name', 'fua_code', 'code_2021', 'class_2021', 'prod_date', 'identifier', 'perimeter', 'area', 'comment']
gdf | rows: 90323 | CRS: EPSG:3035 | columns: ['country', 'fua_name', 'fua_code', 'code_2021', 'class_2021', 'prod_date', 'identifier', 'perimeter', 'area', 'comment']
urban_atlas | rows: 790454 | CRS: EPSG:3035 | columns: ['country', 'fua_name', 'fua_code', 'code_2021', 'class_2021', 'prod_date', 'identifier', 'perimeter', 'area', 'comment']
sample_street_trees | rows: 25375 | CRS: EPSG:3035 | columns: ['area', 'perimeter', 'country', 'fua_code', 'fua_name', 'STL', 'geometry']
street_trees | rows: 532898 | CRS: EPSG:3035 | columns: ['area', 'perimeter', 'country', 'fua_code', 'fua_name', 'STL', 'geometry', 'class']
city_gdf | rows: 90323 | CRS: EPSG:3035 | columns: ['geometry']
heatwaves | rows: 154 | CRS: EPSG:3035 | columns: ['OBJECTID', 'Id', 'hw85_20_52', 'Shape_Leng', 'Shape_Area', 'geometry']
intensity | ro

## City Boundaries for Spatial Analysis

Urban Atlas polygons are dissolved into one boundary per study city.

These boundaries are then used to extract the local LST observations inside each of the ten study areas.

In [96]:
# Create one spatial boundary for each study city
city_boundaries = (
    urban_atlas[["fua_name", "geometry"]]
    .dissolve(by="fua_name")
    .reset_index()
)

print("Number of city boundaries:", len(city_boundaries))
print(city_boundaries["fua_name"].tolist())
print("CRS:", city_boundaries.crs)

/opt/miniconda3/lib/python3.13/site-packages/shapely/set_operations.py:553: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


Number of city boundaries: 10
['Amsterdam', 'Athina', 'Barcelona', 'Berlin', 'Budapest', 'Lisboa', 'Madrid', 'Milano', 'Praha', 'Wien']
CRS: EPSG:3035


## City-Level LST Extraction

LST observations are extracted within the boundaries of the ten study cities.

Each valid grid cell represents a local surface-temperature observation, preserving spatial variation within cities for the subsequent statistical analysis.

In [97]:
from shapely.geometry import Point
import pandas as pd
import geopandas as gpd
import numpy as np

# Convert city boundaries to the same CRS as the LST coordinates
city_boundaries_wgs84 = city_boundaries.to_crs("EPSG:4326")

lst_city_parts = []

for _, row in city_boundaries_wgs84.iterrows():
    city = row["fua_name"]
    geom = row.geometry

    # Use the city's bounding box first to reduce the amount of data
    minx, miny, maxx, maxy = geom.bounds

    city_lst = lst_c.sel(
        lon=slice(minx, maxx),
        lat=slice(miny, maxy)
    )

    df_city = city_lst.to_dataframe().reset_index().dropna(subset=["lst_c"])

    # Keep only LST cells whose centres are inside the city boundary
    points = gpd.GeoDataFrame(
        df_city,
        geometry=gpd.points_from_xy(df_city["lon"], df_city["lat"]),
        crs="EPSG:4326"
    )

    points = points[points.geometry.within(geom)].copy()
    points["city"] = city

    lst_city_parts.append(
        points[["city", "lat", "lon", "lst_c", "geometry"]]
    )

    print(city, ":", len(points), "LST cells")

lst_city = pd.concat(lst_city_parts, ignore_index=True)

print("\nTotal city LST observations:", len(lst_city))
print("\nObservations by city:")
print(lst_city.groupby("city").size())

Amsterdam : 3355 LST cells
Athina : 1929 LST cells
Barcelona : 2803 LST cells
Berlin : 11706 LST cells
Budapest : 7560 LST cells
Lisboa : 4356 LST cells
Madrid : 8326 LST cells
Milano : 3568 LST cells
Praha : 7200 LST cells
Wien : 11448 LST cells

Total city LST observations: 62251

Observations by city:
city
Amsterdam     3355
Athina        1929
Barcelona     2803
Berlin       11706
Budapest      7560
Lisboa        4356
Madrid        8326
Milano        3568
Praha         7200
Wien         11448
dtype: int64


## Local Green-Space Exposure

Local green-space availability is calculated around each LST observation.

This creates a spatial measure that can be directly compared with local surface temperature and used to test the green-space hypothesis.

In [98]:
# Urban Atlas green-space classes
green_codes = ["14110", "14120", "14130", "31000", "32000"]

green_areas = urban_atlas[
    urban_atlas["code_2021"].astype(str).isin(green_codes)
][["fua_name", "geometry"]].copy()

print("Green-space polygons:", len(green_areas))
print("\nBy city:")
print(green_areas.groupby("fua_name").size())

Green-space polygons: 87994

By city:
fua_name
Amsterdam     3794
Athina        8519
Barcelona     8206
Berlin       13064
Budapest      4526
Lisboa       12800
Madrid       15600
Milano        4418
Praha         9296
Wien          7771
dtype: int64


## Spatial Evidence Grid

Each LST observation is represented by its local grid cell. Green-space coverage is calculated within the same spatial unit, allowing temperature and urban characteristics to be compared at a consistent local scale.

In [99]:
# Project LST points to EPSG:3035
lst_grid = gpd.GeoDataFrame(
    lst_city.copy(),
    geometry="geometry",
    crs="EPSG:4326"
).to_crs("EPSG:3035")

# Create approximately 1 km cells around LST centres
lst_grid["geometry"] = lst_grid.geometry.buffer(500).envelope
lst_grid["cell_id"] = np.arange(len(lst_grid))

# Match green-space polygons to LST cells
green_local = gpd.overlay(
    lst_grid[["cell_id", "city", "lst_c", "geometry"]],
    green_areas[["geometry"]],
    how="intersection"
)

green_local["green_area_m2"] = green_local.geometry.area

green_share = (
    green_local.groupby("cell_id")["green_area_m2"]
    .sum()
)

lst_grid["green_space_share"] = (
    lst_grid["cell_id"].map(green_share).fillna(0)
    / lst_grid.geometry.area
).clip(0, 1)

print("Spatial observations:", len(lst_grid))

print("\nGreen-space share:")
print(lst_grid["green_space_share"].describe())

print("\nMean green-space share by city:")
print(
    lst_grid.groupby("city")["green_space_share"]
    .mean()
    .sort_values(ascending=False)
)

/opt/miniconda3/lib/python3.13/site-packages/shapely/set_operations.py:168: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)


Spatial observations: 62251

Green-space share:
count    62251.000000
mean         0.302242
std          0.334385
min          0.000000
25%          0.016750
50%          0.148512
75%          0.546766
max          1.000000
Name: green_space_share, dtype: float64

Mean green-space share by city:
city
Barcelona    0.482279
Athina       0.474077
Berlin       0.398950
Madrid       0.373256
Lisboa       0.370440
Praha        0.276590
Budapest     0.239438
Wien         0.234969
Amsterdam    0.098108
Milano       0.094277
Name: green_space_share, dtype: float64


## Hypothesis 1: Green Space and Surface Temperature

**H0:** Local green-space availability is not associated with local land surface temperature.

**H1:** Higher local green-space availability is associated with lower land surface temperature.

Spearman correlation is used to test the direction and strength of the association.

In [100]:
from scipy.stats import spearmanr

h1_data = lst_grid[
    ["lst_c", "green_space_share"]
].dropna()

rho_green, p_green = spearmanr(
    h1_data["green_space_share"],
    h1_data["lst_c"]
)

print("Observations:", len(h1_data))
print("Spearman rho:", round(rho_green, 4))
print("P-value:", p_green)

if p_green < 0.05:
    print("Result: Reject H0")
else:
    print("Result: Fail to reject H0")

Observations: 62251
Spearman rho: -0.2343
P-value: 0.0
Result: Reject H0


## Local Built-up Surface

Built-up land coverage is derived from Urban Atlas within the same spatial cells used for the LST analysis.

This allows local built-up intensity to be compared directly with local surface temperature.

In [101]:
classes = (
    urban_atlas[["code_2021", "class_2021"]]
    .drop_duplicates()
    .sort_values("code_2021")
)

print(classes.to_string(index=False))

code_2021                                                                      class_2021
    11100                                          Continuous urban fabric (S.L. : > 80%)
    11210                            Discontinuous dense urban fabric (S.L. : 50% -  80%)
    11220                    Discontinuous medium density urban fabric (S.L. : 30% - 50%)
    11230                       Discontinuous low density urban fabric (S.L. : 10% - 30%)
    11240                      Discontinuous very low density urban fabric (S.L. : < 10%)
    11300                                                             Isolated structures
    12100                      Industrial, commercial, public, military and private units
    12210                                          Fast transit roads and associated land
    12220                                                 Other roads and associated land
    12230                                                    Railways and associated land
    12300 

## Local Built-up Share

Built-up land is derived from Urban Atlas classes representing urban fabric, isolated structures, industrial and commercial areas, transport infrastructure, ports, and airports.

The built-up share is calculated within the same spatial cells used for the LST and green-space analysis.

In [102]:
built_codes = [
    "11100", "11210", "11220", "11230", "11240",
    "11300", "12100", "12210", "12220", "12230",
    "12300", "12400"
]

built_areas = urban_atlas[
    urban_atlas["code_2021"].astype(str).isin(built_codes)
][["geometry"]].copy()

# Intersect built-up polygons with LST cells
built_local = gpd.overlay(
    lst_grid[["cell_id", "city", "lst_c", "geometry"]],
    built_areas,
    how="intersection"
)

built_local["built_area_m2"] = built_local.geometry.area

built_share = (
    built_local.groupby("cell_id")["built_area_m2"]
    .sum()
)

lst_grid["built_up_share"] = (
    lst_grid["cell_id"].map(built_share).fillna(0)
    / lst_grid.geometry.area
).clip(0, 1)

print("Built-up polygons:", len(built_areas))

print("\nBuilt-up share:")
print(lst_grid["built_up_share"].describe())

print("\nMean built-up share by city:")
print(
    lst_grid.groupby("city")["built_up_share"]
    .mean()
    .sort_values(ascending=False)
)

/opt/miniconda3/lib/python3.13/site-packages/shapely/set_operations.py:168: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)


Built-up polygons: 519439

Built-up share:
count    62251.000000
mean         0.174996
std          0.246683
min          0.000000
25%          0.008842
50%          0.054173
75%          0.241436
max          1.000000
Name: built_up_share, dtype: float64

Mean built-up share by city:
city
Milano       0.336738
Athina       0.308046
Amsterdam    0.273266
Barcelona    0.260968
Lisboa       0.182748
Budapest     0.169784
Berlin       0.160443
Madrid       0.145475
Praha        0.145112
Wien         0.107957
Name: built_up_share, dtype: float64


## Hypothesis 2: Built-up Surface and Surface Temperature

**H0:** Local built-up share is not associated with local land surface temperature.

**H1:** Higher local built-up share is associated with higher local land surface temperature.

Spearman correlation is used to test the direction and strength of the association.

In [103]:
from scipy.stats import spearmanr

h2_data = lst_grid[
    ["lst_c", "built_up_share"]
].dropna()

rho_built, p_built = spearmanr(
    h2_data["built_up_share"],
    h2_data["lst_c"]
)

print("Observations:", len(h2_data))
print("Spearman rho:", round(rho_built, 4))
print("P-value:", f"{p_built:.3e}")

if p_built < 0.05:
    print("Result: Reject H0")
else:
    print("Result: Fail to reject H0")

Observations: 62251
Spearman rho: 0.0598
P-value: 1.761e-50
Result: Reject H0


## Spatial Aggregation for Statistical Testing

To reduce spatial dependence between neighbouring LST observations, the local 1 km cells are aggregated into approximately 2 km spatial blocks.

Mean LST, green-space share, and built-up share are calculated for each block before inferential testing.

In [104]:
# Use cell centres to create approximately 2 km spatial blocks
cell_centres = lst_grid.geometry.centroid

lst_grid["block_x"] = (cell_centres.x // 2000).astype(int)
lst_grid["block_y"] = (cell_centres.y // 2000).astype(int)

spatial_evidence = (
    lst_grid.groupby(["city", "block_x", "block_y"], as_index=False)
    .agg(
        lst=("lst_c", "mean"),
        green_space_share=("green_space_share", "mean"),
        built_up_share=("built_up_share", "mean"),
        n_cells=("cell_id", "count")
    )
)

print("Original observations:", len(lst_grid))
print("Aggregated spatial blocks:", len(spatial_evidence))

print("\nBlocks by city:")
print(spatial_evidence.groupby("city").size())

Original observations: 62251
Aggregated spatial blocks: 14176

Blocks by city:
city
Amsterdam     732
Athina        516
Barcelona     707
Berlin       2364
Budapest     1704
Lisboa       1145
Madrid       2096
Milano        847
Praha        1561
Wien         2504
dtype: int64


## Hypothesis Testing After Spatial Aggregation

Both hypotheses are tested again using the aggregated spatial blocks to reduce dependence between neighbouring observations.

A significance level of 0.05 is used.

In [105]:
from scipy.stats import spearmanr

# H1: Green space vs LST
rho_green_2km, p_green_2km = spearmanr(
    spatial_evidence["green_space_share"],
    spatial_evidence["lst"]
)

# H2: Built-up share vs LST
rho_built_2km, p_built_2km = spearmanr(
    spatial_evidence["built_up_share"],
    spatial_evidence["lst"]
)

print("Observations:", len(spatial_evidence))

print("\nH1: Green Space vs LST")
print("Spearman rho:", round(rho_green_2km, 4))
print("P-value:", f"{p_green_2km:.3e}")

print("\nH2: Built-up Share vs LST")
print("Spearman rho:", round(rho_built_2km, 4))
print("P-value:", f"{p_built_2km:.3e}")

Observations: 14176

H1: Green Space vs LST
Spearman rho: -0.2092
P-value: 6.529e-140

H2: Built-up Share vs LST
Spearman rho: 0.0553
P-value: 4.409e-11


## Multivariable Regression with City Controls

A multivariable regression model is used to test whether local green-space and built-up shares remain associated with LST after accounting for systematic differences between cities.

City fixed effects control for baseline differences between the ten study areas. Green-space and built-up shares are included simultaneously so that each coefficient is estimated while holding the other factor constant.

In [106]:
import statsmodels.formula.api as smf

reg_data = spatial_evidence.copy()

# Coefficients represent a 10 percentage-point increase
reg_data["green_10pp"] = reg_data["green_space_share"] / 0.10
reg_data["built_10pp"] = reg_data["built_up_share"] / 0.10

model = smf.ols(
    "lst ~ green_10pp + built_10pp + C(city)",
    data=reg_data
).fit(cov_type="HC3")

print("Observations:", int(model.nobs))
print("R-squared:", round(model.rsquared, 4))

print("\nGreen space:")
print("Coefficient:", round(model.params["green_10pp"], 4))
print("P-value:", f'{model.pvalues["green_10pp"]:.3e}')

print("\nBuilt-up:")
print("Coefficient:", round(model.params["built_10pp"], 4))
print("P-value:", f'{model.pvalues["built_10pp"]:.3e}')

Observations: 14176
R-squared: 0.8128

Green space:
Coefficient: -0.5977
P-value: 0.000e+00

Built-up:
Coefficient: -0.0029
P-value: 7.748e-01


## Green Space and LST by City

The green-space and LST relationship is examined separately within each study area to assess whether the overall association is consistent across cities.

In [107]:
from scipy.stats import spearmanr
import pandas as pd

city_results = []

for city, group in spatial_evidence.groupby("city"):
    rho, p = spearmanr(
        group["green_space_share"],
        group["lst"]
    )

    city_results.append({
        "city": city,
        "observations": len(group),
        "rho": rho,
        "p_value": p
    })

city_green_results = pd.DataFrame(city_results)

print(
    city_green_results
    .sort_values("rho")
    .round({"rho": 3, "p_value": 4})
    .to_string(index=False)
)

     city  observations    rho  p_value
Barcelona           707 -0.850   0.0000
    Praha          1561 -0.788   0.0000
     Wien          2504 -0.706   0.0000
 Budapest          1704 -0.674   0.0000
   Berlin          2364 -0.667   0.0000
   Madrid          2096 -0.407   0.0000
   Athina           516 -0.370   0.0000
Amsterdam           732 -0.010   0.7938
   Milano           847  0.020   0.5705
   Lisboa          1145  0.089   0.0027


In [108]:
# Find population-related objects already loaded in the notebook
for name, obj in globals().items():
    if "pop" in name.lower():
        try:
            print(name, "|", type(obj).__name__, "|", getattr(obj, "shape", ""))
        except:
            pass

POPULATION_PATH | PosixPath | 
population_raster | PosixPath | 
population_by_city | Series | (10,)
population_array | ndarray | (1299, 1190)
population_indicators | DataFrame | (10, 2)


In [109]:
for name, obj in globals().items():
    if any(word in name.lower() for word in ["transform", "profile", "meta", "crs", "population"]):
        try:
            if not callable(obj):
                print(name, "|", type(obj).__name__)
        except:
            pass

POPULATION_PATH | PosixPath
population_raster | PosixPath
population_by_city | Series
population_array | ndarray
window_transform | Affine
population_indicators | DataFrame
transformer | Transformer


In [110]:
print("Population array shape:", population_array.shape)
print("Window transform:", window_transform)

print("\nPopulation raster path:")
print(population_raster)

Population array shape: (1299, 1190)
Window transform: | 100.00, 0.00, 4735600.00|
| 0.00,-100.00, 2879700.00|
| 0.00, 0.00, 1.00|

Population raster path:
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_raw/population/JRC-ESTAT_Census_Population_2021_100m_rev0726.tif


## Local Population Exposure

Population data from the JRC 2021 100 m raster is spatially aggregated to the same analysis blocks used for the heat analysis.

This allows local population exposure to be tested against land surface temperature.

In [111]:
import rasterio
from rasterio.mask import mask
import numpy as np

# Create 2 km block geometries
blocks_gdf = lst_grid.copy()

blocks_gdf["block_x"] = (
    blocks_gdf.geometry.centroid.x // 2000
).astype(int)

blocks_gdf["block_y"] = (
    blocks_gdf.geometry.centroid.y // 2000
).astype(int)

blocks_gdf = (
    blocks_gdf
    .dissolve(by=["city", "block_x", "block_y"])
    .reset_index()
)

population_values = []

with rasterio.open(population_raster) as src:

    blocks_population = blocks_gdf.to_crs(src.crs)

    for geom in blocks_population.geometry:
        try:
            data, _ = mask(
                src,
                [geom],
                crop=True,
                filled=False
            )

            values = data[0]

            if np.ma.isMaskedArray(values):
                total_population = values.sum()
                total_population = (
                    float(total_population)
                    if not np.ma.is_masked(total_population)
                    else 0
                )
            else:
                total_population = float(np.nansum(values))

            population_values.append(total_population)

        except ValueError:
            population_values.append(0)

blocks_population["population"] = population_values

print("Spatial blocks:", len(blocks_population))

print("\nPopulation summary:")
print(blocks_population["population"].describe())

print("\nMean population per block by city:")
print(
    blocks_population.groupby("city")["population"]
    .mean()
    .sort_values(ascending=False)
)

/opt/miniconda3/lib/python3.13/site-packages/shapely/set_operations.py:553: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


Spatial blocks: 14176

Population summary:
count     14176.000000
mean       2776.364207
std        9208.723834
min           0.000000
25%           6.000000
50%         190.000000
75%        1231.000000
max      176429.000000
Name: population, dtype: float64

Mean population per block by city:
city
Barcelona    6867.308345
Athina       6528.668605
Milano       5807.088548
Amsterdam    4035.565574
Madrid       3265.530057
Lisboa       2453.597380
Berlin       2150.524535
Budapest     1805.437793
Praha        1506.989110
Wien         1235.816693
Name: population, dtype: float64


## Hypothesis 3: Population and Surface Temperature

**H0:** Local population concentration is not associated with local land surface temperature.

**H1:** Higher local population concentration is associated with higher local land surface temperature.

Population is aggregated to the same spatial blocks used for the heat analysis.

In [112]:
# Add population to the spatial evidence table
population_blocks = blocks_population[
    ["city", "block_x", "block_y", "population"]
].copy()

spatial_evidence = spatial_evidence.merge(
    population_blocks,
    on=["city", "block_x", "block_y"],
    how="left"
)

spatial_evidence["population"] = (
    spatial_evidence["population"].fillna(0)
)

rho_pop, p_pop = spearmanr(
    spatial_evidence["population"],
    spatial_evidence["lst"]
)

print("Observations:", len(spatial_evidence))
print("Spearman rho:", round(rho_pop, 4))
print("P-value:", f"{p_pop:.3e}")

Observations: 14176
Spearman rho: -0.0144
P-value: 8.656e-02


## Final Multivariable Model

Green space, built-up land, and local population are tested simultaneously while controlling for systematic differences between cities.

Population is log-transformed because its spatial distribution is highly skewed.

In [113]:
import numpy as np
import statsmodels.formula.api as smf

final_model_data = spatial_evidence.copy()

final_model_data["green_10pp"] = (
    final_model_data["green_space_share"] / 0.10
)

final_model_data["built_10pp"] = (
    final_model_data["built_up_share"] / 0.10
)

final_model_data["log_population"] = np.log1p(
    final_model_data["population"]
)

final_model = smf.ols(
    "lst ~ green_10pp + built_10pp + log_population + C(city)",
    data=final_model_data
).fit(cov_type="HC3")

print("Observations:", int(final_model.nobs))
print("R-squared:", round(final_model.rsquared, 4))

print("\nGreen space")
print("Coefficient:", round(final_model.params["green_10pp"], 4))
print("P-value:", f'{final_model.pvalues["green_10pp"]:.3e}')

print("\nBuilt-up")
print("Coefficient:", round(final_model.params["built_10pp"], 4))
print("P-value:", f'{final_model.pvalues["built_10pp"]:.3e}')

print("\nPopulation")
print("Coefficient:", round(final_model.params["log_population"], 4))
print("P-value:", f'{final_model.pvalues["log_population"]:.3e}')

Observations: 14176
R-squared: 0.8139

Green space
Coefficient: -0.6061
P-value: 0.000e+00

Built-up
Coefficient: 0.088
P-value: 3.134e-11

Population
Coefficient: -0.0895
P-value: 1.482e-19


## Water Coverage and Surface Temperature

Water bodies can contribute to local urban cooling through evaporation and their thermal properties.

Water coverage is calculated for the same spatial blocks used in the LST analysis and tested for its association with surface temperature.

In [114]:
# Select water polygons from Urban Atlas
water_codes = ["50000"]

water_polygons = urban_atlas[
    urban_atlas["code_2021"].astype(str).isin(water_codes)
][["geometry"]].copy()

# Make sure both layers use the same CRS
water_polygons = water_polygons.to_crs(lst_grid.crs)

# Intersect water polygons with the local LST grid
water_intersection = gpd.overlay(
    lst_grid[["cell_id", "geometry"]],
    water_polygons,
    how="intersection"
)

# Calculate intersected water area
water_intersection["water_area"] = (
    water_intersection.geometry.area
)

water_area_by_cell = (
    water_intersection
    .groupby("cell_id")["water_area"]
    .sum()
)

# Add water area to LST grid
lst_grid["water_area"] = (
    lst_grid["cell_id"]
    .map(water_area_by_cell)
    .fillna(0)
)

# Calculate share of each cell covered by water
lst_grid["cell_area"] = lst_grid.geometry.area

lst_grid["water_share"] = (
    lst_grid["water_area"] / lst_grid["cell_area"]
).clip(0, 1)

print("Water polygons:", len(water_polygons))

print("\nWater share summary:")
print(lst_grid["water_share"].describe())

print("\nMean water share by city:")
print(
    lst_grid.groupby("city")["water_share"]
    .mean()
    .sort_values(ascending=False)
)

Water polygons: 7863

Water share summary:
count    62251.000000
mean         0.014139
std          0.058166
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: water_share, dtype: float64

Mean water share by city:
city
Amsterdam    0.064337
Berlin       0.018084
Budapest     0.015160
Lisboa       0.014578
Praha        0.010346
Milano       0.010158
Wien         0.008639
Athina       0.007397
Barcelona    0.005085
Madrid       0.004364
Name: water_share, dtype: float64


/opt/miniconda3/lib/python3.13/site-packages/shapely/set_operations.py:168: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)


## Hypothesis 4: Water Coverage and Surface Temperature

**H0:** Local water coverage is not associated with local land surface temperature.

**H1:** Higher local water coverage is associated with lower local land surface temperature.

Water coverage is aggregated to the same spatial blocks used for the other urban factors.

In [115]:
# Aggregate water coverage to the same 2 km spatial blocks
water_blocks = (
    lst_grid
    .groupby(["city", "block_x", "block_y"], as_index=False)
    .agg(
        water_share=("water_share", "mean")
    )
)

# Add water coverage to the evidence table
spatial_evidence = spatial_evidence.merge(
    water_blocks,
    on=["city", "block_x", "block_y"],
    how="left"
)

spatial_evidence["water_share"] = (
    spatial_evidence["water_share"].fillna(0)
)

# Test association between water coverage and LST
rho_water, p_water = spearmanr(
    spatial_evidence["water_share"],
    spatial_evidence["lst"]
)

print("Observations:", len(spatial_evidence))
print("Blocks with water:", (spatial_evidence["water_share"] > 0).sum())
print("Spearman rho:", round(rho_water, 4))
print("P-value:", f"{p_water:.3e}")

Observations: 14176
Blocks with water: 4865
Spearman rho: -0.2678
P-value: 2.113e-231


## Multivariable Model Including Water

Water coverage is added to the multivariable model to test whether its association with surface temperature remains after controlling for green space, built-up land, population, and city-level differences.

In [116]:
water_model_data = spatial_evidence.copy()

water_model_data["green_10pp"] = (
    water_model_data["green_space_share"] / 0.10
)

water_model_data["built_10pp"] = (
    water_model_data["built_up_share"] / 0.10
)

water_model_data["water_10pp"] = (
    water_model_data["water_share"] / 0.10
)

water_model_data["log_population"] = np.log1p(
    water_model_data["population"]
)

water_model = smf.ols(
    "lst ~ green_10pp + built_10pp + water_10pp + log_population + C(city)",
    data=water_model_data
).fit(cov_type="HC3")

print("Observations:", int(water_model.nobs))
print("R-squared:", round(water_model.rsquared, 4))

print("\nGreen space")
print("Coefficient:", round(water_model.params["green_10pp"], 4))
print("P-value:", f'{water_model.pvalues["green_10pp"]:.3e}')

print("\nBuilt-up")
print("Coefficient:", round(water_model.params["built_10pp"], 4))
print("P-value:", f'{water_model.pvalues["built_10pp"]:.3e}')

print("\nWater")
print("Coefficient:", round(water_model.params["water_10pp"], 4))
print("P-value:", f'{water_model.pvalues["water_10pp"]:.3e}')

print("\nPopulation")
print("Coefficient:", round(water_model.params["log_population"], 4))
print("P-value:", f'{water_model.pvalues["log_population"]:.3e}')

Observations: 14176
R-squared: 0.8189

Green space
Coefficient: -0.6121
P-value: 0.000e+00

Built-up
Coefficient: 0.0873
P-value: 3.790e-11

Water
Coefficient: -0.7921
P-value: 1.668e-92

Population
Coefficient: -0.09
P-value: 3.722e-20


In [117]:
print("Street tree rows:", len(street_trees))
print("CRS:", street_trees.crs)

print("\nColumns:")
print(street_trees.columns.tolist())

print("\nGeometry types:")
print(street_trees.geometry.geom_type.value_counts())

print("\nFirst rows:")
print(street_trees.head())

Street tree rows: 532898
CRS: EPSG:3035

Columns:
['area', 'perimeter', 'country', 'fua_code', 'fua_name', 'STL', 'geometry', 'class']

Geometry types:
MultiPolygon    453899
Polygon          78999
Name: count, dtype: int64

First rows:
     area   perimeter country fua_code fua_name  STL  \
0   537.5  136.148276      EL  EL001L1   ATHINA  1.0   
1   600.0  123.027756      EL  EL001L1   ATHINA  1.0   
2  1362.5  163.087106      EL  EL001L1   ATHINA  1.0   
3  1225.0  172.503869      EL  EL001L1   ATHINA  1.0   
4  1187.5  146.534064      EL  EL001L1   ATHINA  1.0   

                                            geometry  class  
0  MULTIPOLYGON (((5563503.876 1737212.827, 55634...    NaN  
1  MULTIPOLYGON (((5563618.876 1737352.827, 55636...    NaN  
2  MULTIPOLYGON (((5562928.876 1737482.827, 55629...    NaN  
3  MULTIPOLYGON (((5562998.876 1737307.827, 55630...    NaN  
4  MULTIPOLYGON (((5562898.876 1737182.827, 55629...    NaN  


In [118]:
# Intersect street-tree polygons with the LST grid
tree_intersection = gpd.overlay(
    lst_grid[["cell_id", "geometry"]],
    street_trees[["geometry"]],
    how="intersection"
)

# Calculate tree area inside each cell
tree_intersection["tree_area"] = tree_intersection.geometry.area

tree_area_by_cell = (
    tree_intersection
    .groupby("cell_id")["tree_area"]
    .sum()
)

# Add tree coverage to the LST grid
lst_grid["tree_area"] = (
    lst_grid["cell_id"]
    .map(tree_area_by_cell)
    .fillna(0)
)

lst_grid["street_tree_share"] = (
    lst_grid["tree_area"] / lst_grid.geometry.area
).clip(0, 1)

print("Intersected tree polygons:", len(tree_intersection))

print("\nStreet-tree share summary:")
print(lst_grid["street_tree_share"].describe())

print("\nMean street-tree share by city:")
print(
    lst_grid.groupby("city")["street_tree_share"]
    .mean()
    .sort_values(ascending=False)
)

/opt/miniconda3/lib/python3.13/site-packages/shapely/set_operations.py:168: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)


Intersected tree polygons: 754731

Street-tree share summary:
count    62251.000000
mean         0.215868
std          0.307540
min          0.000000
25%          0.000000
50%          0.059575
75%          0.305940
max          1.000000
Name: street_tree_share, dtype: float64

Mean street-tree share by city:
city
Milano       0.400508
Amsterdam    0.348950
Athina       0.346785
Barcelona    0.302738
Berlin       0.213405
Budapest     0.213249
Lisboa       0.208354
Praha        0.185128
Madrid       0.180237
Wien         0.128345
Name: street_tree_share, dtype: float64


## Hypothesis 5: Street Trees and Surface Temperature

**H0:** Local street-tree coverage is not associated with local land surface temperature.

**H1:** Higher local street-tree coverage is associated with lower local land surface temperature.

In [119]:
# Aggregate street-tree coverage to the same 2 km spatial blocks
tree_blocks = (
    lst_grid
    .groupby(["city", "block_x", "block_y"], as_index=False)
    .agg(
        street_tree_share=("street_tree_share", "mean")
    )
)

# Add street-tree coverage to the evidence table
spatial_evidence = spatial_evidence.merge(
    tree_blocks,
    on=["city", "block_x", "block_y"],
    how="left"
)

spatial_evidence["street_tree_share"] = (
    spatial_evidence["street_tree_share"].fillna(0)
)

# Test association with LST
rho_tree, p_tree = spearmanr(
    spatial_evidence["street_tree_share"],
    spatial_evidence["lst"]
)

print("Observations:", len(spatial_evidence))
print("Blocks with street trees:",
      (spatial_evidence["street_tree_share"] > 0).sum())
print("Spearman rho:", round(rho_tree, 4))
print("P-value:", f"{p_tree:.3e}")

Observations: 14176
Blocks with street trees: 12693
Spearman rho: 0.0287
P-value: 6.410e-04


## Final Multivariable Urban Heat Model

All urban factors are tested simultaneously while controlling for population and systematic differences between cities.

This model identifies which factors retain an independent association with land surface temperature.

In [120]:
final_heat_data = spatial_evidence.copy()

final_heat_data["green_10pp"] = (
    final_heat_data["green_space_share"] / 0.10
)

final_heat_data["built_10pp"] = (
    final_heat_data["built_up_share"] / 0.10
)

final_heat_data["water_10pp"] = (
    final_heat_data["water_share"] / 0.10
)

final_heat_data["trees_10pp"] = (
    final_heat_data["street_tree_share"] / 0.10
)

final_heat_data["log_population"] = np.log1p(
    final_heat_data["population"]
)

final_heat_model = smf.ols(
    "lst ~ green_10pp + built_10pp + water_10pp + trees_10pp + log_population + C(city)",
    data=final_heat_data
).fit(cov_type="HC3")

print("Observations:", int(final_heat_model.nobs))
print("R-squared:", round(final_heat_model.rsquared, 4))

for variable, label in [
    ("green_10pp", "Green space"),
    ("water_10pp", "Water"),
    ("built_10pp", "Built-up"),
    ("trees_10pp", "Street trees"),
    ("log_population", "Population")
]:
    print(f"\n{label}")
    print("Coefficient:",
          round(final_heat_model.params[variable], 4))
    print("P-value:",
          f'{final_heat_model.pvalues[variable]:.3e}')

Observations: 14176
R-squared: 0.8191

Green space
Coefficient: -0.6089
P-value: 0.000e+00

Water
Coefficient: -0.7742
P-value: 1.565e-87

Built-up
Coefficient: 0.198
P-value: 3.722e-08

Street trees
Coefficient: -0.0949
P-value: 9.890e-04

Population
Coefficient: -0.0847
P-value: 3.092e-17


In [121]:
print("Building rows:", len(buildings_inside_city))
print("CRS:", buildings_inside_city.crs)

print("\nColumns:")
print(buildings_inside_city.columns.tolist())

print("\nGeometry types:")
print(buildings_inside_city.geometry.geom_type.value_counts())

print("\nFirst rows:")
print(buildings_inside_city.head())

Building rows: 9439088
CRS: EPSG:3035

Columns:
['id', 'lon', 'lat', 'height', 'shapefactor', 'use', 'epoch', 'area', 'perimeter', 'city', 'geometry']

Geometry types:
Point    9439088
Name: count, dtype: int64

First rows:
                                 id      lon       lat  height  shapefactor  \
0  08b1968218234fff020029bc01e6941d  4.53563  52.45952     NaN          NaN   
1  08b1968218343fff02001b82305962e3  4.54073  52.45834     2.5      1.29933   
2  08b1968218341fff0200a4fb5f6fa701  4.54156  52.45857     2.5      1.59301   
3  08b196821836afff020081d39b3e4bc5  4.54187  52.45859     2.5      1.32363   
4  08b19682180edfff0200b5fcb988097c  4.53303  52.46203     NaN          NaN   

   use  epoch    area  perimeter       city                         geometry  
0    0      0  122.17      67.06  Amsterdam    POINT (3949970.17 3275070.26)  
1    1      1  242.84     121.26  Amsterdam  POINT (3950305.816 3274913.273)  
2    1      1   52.55      41.67  Amsterdam  POINT (3950363.979 

In [122]:
print("Total buildings:", len(buildings_inside_city))

print(
    "Buildings with height:",
    buildings_inside_city["height"].notna().sum()
)

print(
    "Missing height:",
    buildings_inside_city["height"].isna().sum()
)

print("\nHeight summary:")
print(buildings_inside_city["height"].describe())

print("\nBuildings with height by city:")
print(
    buildings_inside_city
    .groupby("city")["height"]
    .agg(["count", "median", "mean", "max"])
)

Total buildings: 9439088
Buildings with height: 9352238
Missing height: 86850

Height summary:
count    9.352238e+06
mean     6.775901e+00
std      4.272909e+00
min      2.500000e+00
25%      2.980000e+00
50%      6.020000e+00
75%      8.830000e+00
max      8.934000e+01
Name: height, dtype: float64

Buildings with height by city:
             count  median       mean    max
city                                        
Amsterdam  1508430    7.21   7.356376  72.47
Athina      668710   10.64   9.769938  39.81
Barcelona   470040   10.47  10.610237  37.96
Berlin     1702036    4.99   5.459626  89.34
Budapest   1102587    5.01   5.231145  34.34
Lisboa      751888    5.63   6.952854  43.08
Madrid      522056    7.46   8.078552  43.35
Milano      731046    8.74   9.383429  59.99
Praha       897048    3.37   4.772399  28.78
Wien        998397    3.50   5.114686  43.52


## Building Morphology and Surface Temperature

Building height and building density are analysed as additional urban-form characteristics using the existing GHS-OBAT building data.

Both indicators are calculated for the same spatial blocks used in the LST analysis.

In [123]:
# Assign each building to the 2 km analysis grid
building_blocks = gpd.sjoin(
    buildings_inside_city[
        ["city", "height", "geometry"]
    ],
    blocks_gdf[
        ["city", "block_x", "block_y", "geometry"]
    ],
    how="inner",
    predicate="within"
)

# Keep buildings matched to the correct city
building_blocks = building_blocks[
    building_blocks["city_left"] == building_blocks["city_right"]
].copy()

# Calculate building indicators for each spatial block
building_spatial = (
    building_blocks
    .groupby(
        ["city_left", "block_x", "block_y"],
        as_index=False
    )
    .agg(
        building_count=("geometry", "count"),
        mean_building_height=("height", "mean")
    )
    .rename(columns={"city_left": "city"})
)

print("Spatial blocks with buildings:", len(building_spatial))

print("\nBuilding count summary:")
print(building_spatial["building_count"].describe())

print("\nMean building height summary:")
print(building_spatial["mean_building_height"].describe())

print("\nMean building height by city:")
print(
    building_spatial.groupby("city")["mean_building_height"]
    .mean()
    .sort_values(ascending=False)
)

Spatial blocks with buildings: 12847

Building count summary:
count    12847.000000
mean       737.725228
std       1309.643178
min          1.000000
25%         39.000000
50%        261.000000
75%        846.000000
max      19693.000000
Name: building_count, dtype: float64

Mean building height summary:
count    12291.000000
mean         4.118839
std          2.714936
min          2.500000
25%          2.500000
50%          2.741938
75%          4.689543
max         23.521402
Name: mean_building_height, dtype: float64

Mean building height by city:
city
Milano       6.668811
Barcelona    6.245210
Athina       5.316151
Amsterdam    4.914001
Madrid       4.467794
Lisboa       4.128421
Budapest     3.556713
Berlin       3.525875
Praha        3.383342
Wien         3.129800
Name: mean_building_height, dtype: float64


In [124]:
# Add building indicators to the spatial evidence table
spatial_evidence = spatial_evidence.merge(
    building_spatial[
        ["city", "block_x", "block_y",
         "building_count", "mean_building_height"]
    ],
    on=["city", "block_x", "block_y"],
    how="left"
)

# Blocks without buildings get a building count of zero
spatial_evidence["building_count"] = (
    spatial_evidence["building_count"].fillna(0)
)

# Building count vs LST
rho_count, p_count = spearmanr(
    spatial_evidence["building_count"],
    spatial_evidence["lst"]
)

# Building height vs LST
height_data = spatial_evidence.dropna(
    subset=["mean_building_height", "lst"]
)

rho_height, p_height = spearmanr(
    height_data["mean_building_height"],
    height_data["lst"]
)

print("Building density")
print("Observations:", len(spatial_evidence))
print("Spearman rho:", round(rho_count, 4))
print("P-value:", f"{p_count:.3e}")

print("\nBuilding height")
print("Observations:", len(height_data))
print("Spearman rho:", round(rho_height, 4))
print("P-value:", f"{p_height:.3e}")

Building density
Observations: 14176
Spearman rho: -0.1171
P-value: 1.722e-44

Building height
Observations: 12291
Spearman rho: 0.0687
P-value: 2.413e-14


In [125]:
# Calculate actual block area in km²
block_areas = blocks_gdf[
    ["city", "block_x", "block_y", "geometry"]
].copy()

block_areas["block_area_km2"] = (
    block_areas.geometry.area / 1_000_000
)

# Add block area
spatial_evidence = spatial_evidence.merge(
    block_areas[
        ["city", "block_x", "block_y", "block_area_km2"]
    ],
    on=["city", "block_x", "block_y"],
    how="left"
)

# Calculate true building density
spatial_evidence["building_density_km2"] = (
    spatial_evidence["building_count"]
    / spatial_evidence["block_area_km2"]
)

print(
    spatial_evidence["building_density_km2"].describe()
)

count    14176.000000
mean       169.682364
std        311.201753
min          0.000000
25%          4.830286
50%         52.145720
75%        193.966255
max       4153.381579
Name: building_density_km2, dtype: float64


## Extended Multivariable Urban Heat Model

Building density and average building height are added to the urban heat model to test whether urban morphology provides additional information beyond green space, water, built-up land, street trees and population.

In [126]:
extended_data = spatial_evidence.dropna(
    subset=["mean_building_height"]
).copy()

extended_data["green_10pp"] = (
    extended_data["green_space_share"] / 0.10
)

extended_data["water_10pp"] = (
    extended_data["water_share"] / 0.10
)

extended_data["built_10pp"] = (
    extended_data["built_up_share"] / 0.10
)

extended_data["trees_10pp"] = (
    extended_data["street_tree_share"] / 0.10
)

extended_data["log_population"] = np.log1p(
    extended_data["population"]
)

extended_data["building_density_100"] = (
    extended_data["building_density_km2"] / 100
)

extended_model = smf.ols(
    """
    lst ~ green_10pp
        + water_10pp
        + built_10pp
        + trees_10pp
        + log_population
        + building_density_100
        + mean_building_height
        + C(city)
    """,
    data=extended_data
).fit(cov_type="HC3")

print("Observations:", int(extended_model.nobs))
print("R-squared:", round(extended_model.rsquared, 4))

for variable, label in [
    ("green_10pp", "Green space"),
    ("water_10pp", "Water"),
    ("built_10pp", "Built-up"),
    ("trees_10pp", "Street trees"),
    ("log_population", "Population"),
    ("building_density_100", "Building density"),
    ("mean_building_height", "Building height")
]:
    print(f"\n{label}")
    print(
        "Coefficient:",
        round(extended_model.params[variable], 4)
    )
    print(
        "P-value:",
        f'{extended_model.pvalues[variable]:.3e}'
    )

Observations: 12291
R-squared: 0.8275

Green space
Coefficient: -0.6097
P-value: 0.000e+00

Water
Coefficient: -0.7683
P-value: 7.897e-90

Built-up
Coefficient: 0.1197
P-value: 1.226e-03

Street trees
Coefficient: -0.0954
P-value: 9.877e-04

Population
Coefficient: -0.0988
P-value: 2.821e-17

Building density
Coefficient: -0.0008
P-value: 9.399e-01

Building height
Coefficient: 0.1036
P-value: 3.869e-14


In [127]:
print("Building density P-value:",
      f'{extended_model.pvalues["building_density_100"]:.3e}')

Building density P-value: 9.399e-01


## Heat Hotspots and Population Exposure

Hotspots combine local surface temperature with population exposure to identify areas where urban cooling interventions may have the greatest relevance.

Higher temperature and higher population exposure indicate greater local cooling pressure.

In [128]:
# Create within-city temperature and population percentiles
hotspot_data = spatial_evidence.copy()

hotspot_data["heat_percentile"] = (
    hotspot_data.groupby("city")["lst"]
    .rank(pct=True)
)

hotspot_data["population_percentile"] = (
    hotspot_data.groupby("city")["population"]
    .rank(pct=True)
)

# Combined hotspot score
hotspot_data["hotspot_score"] = (
    hotspot_data["heat_percentile"]
    + hotspot_data["population_percentile"]
) / 2

# Define high-priority hotspots
hotspot_data["hotspot"] = (
    (hotspot_data["heat_percentile"] >= 0.75)
    & (hotspot_data["population_percentile"] >= 0.75)
)

print("Total spatial blocks:", len(hotspot_data))
print("Hotspot blocks:", hotspot_data["hotspot"].sum())

print("\nHotspots by city:")
print(
    hotspot_data.groupby("city")["hotspot"]
    .agg(["sum", "count"])
    .sort_values("sum", ascending=False)
)

Total spatial blocks: 14176
Hotspot blocks: 954

Hotspots by city:
           sum  count
city                 
Berlin     204   2364
Wien       142   2504
Milano     130    847
Praha      108   1561
Amsterdam   94    732
Barcelona   87    707
Madrid      64   2096
Budapest    52   1704
Lisboa      49   1145
Athina      24    516


## Hotspot Exposure by City

Hotspot prevalence and population exposure are calculated for each study area to distinguish the number of hotspots from their relative importance within each city.

In [129]:
city_hotspot_summary = (
    hotspot_data.groupby("city")
    .agg(
        total_blocks=("hotspot", "size"),
        hotspot_blocks=("hotspot", "sum"),
        total_population=("population", "sum")
    )
    .reset_index()
)

hotspot_population = (
    hotspot_data[hotspot_data["hotspot"]]
    .groupby("city")["population"]
    .sum()
    .rename("hotspot_population")
    .reset_index()
)

city_hotspot_summary = city_hotspot_summary.merge(
    hotspot_population,
    on="city",
    how="left"
)

city_hotspot_summary["hotspot_population"] = (
    city_hotspot_summary["hotspot_population"].fillna(0)
)

city_hotspot_summary["hotspot_share_pct"] = (
    city_hotspot_summary["hotspot_blocks"]
    / city_hotspot_summary["total_blocks"]
    * 100
)

city_hotspot_summary["population_exposed_pct"] = (
    city_hotspot_summary["hotspot_population"]
    / city_hotspot_summary["total_population"]
    * 100
)

print(
    city_hotspot_summary[
        [
            "city",
            "hotspot_blocks",
            "hotspot_share_pct",
            "hotspot_population",
            "population_exposed_pct"
        ]
    ]
    .sort_values("hotspot_share_pct", ascending=False)
    .round(2)
)

        city  hotspot_blocks  hotspot_share_pct  hotspot_population  \
7     Milano             130              15.35           2632508.0   
0  Amsterdam              94              12.84           1616346.0   
2  Barcelona              87              12.31           2925716.0   
3     Berlin             204               8.63           3170534.0   
8      Praha             108               6.92            701810.0   
9       Wien             142               5.67            207140.0   
1     Athina              24               4.65            264816.0   
5     Lisboa              49               4.28            274426.0   
6     Madrid              64               3.05            211942.0   
4   Budapest              52               3.05            338123.0   

   population_exposed_pct  
7                   53.52  
0                   54.72  
2                   60.26  
3                   62.36  
8                   29.83  
9                    6.69  
1                    7

## Characteristics of Heat Hotspots

Urban characteristics inside hotspot blocks are compared with non-hotspot blocks to identify the local conditions associated with areas of high heat and population exposure.

In [130]:
hotspot_characteristics = (
    hotspot_data
    .groupby("hotspot")
    .agg(
        mean_lst=("lst", "mean"),
        mean_green=("green_space_share", "mean"),
        mean_water=("water_share", "mean"),
        mean_built=("built_up_share", "mean"),
        mean_trees=("street_tree_share", "mean"),
        mean_population=("population", "mean"),
        mean_building_height=("mean_building_height", "mean")
    )
    .reset_index()
)

for col in [
    "mean_green",
    "mean_water",
    "mean_built",
    "mean_trees"
]:
    hotspot_characteristics[col] *= 100

print(hotspot_characteristics.round(2))

   hotspot   mean_lst  mean_green  mean_water  mean_built  mean_trees  \
0    False  35.610001       31.97        1.61       14.82       18.29   
1     True  37.369999        9.84        1.22       47.40       57.41   

   mean_population  mean_building_height  
0          2043.14                  3.85  
1         12938.53                  7.30  


## Evidence-Based Cooling Recommendations

Cooling recommendations are assigned to hotspot blocks according to local urban conditions and the statistically supported heat factors.

The framework links each hotspot to the intervention most relevant to its local cooling deficit.

In [131]:
# Thresholds based on the distribution of hotspot characteristics
hotspots_only = hotspot_data[hotspot_data["hotspot"]].copy()

green_threshold = hotspots_only["green_space_share"].median()
water_threshold = hotspots_only["water_share"].median()
tree_threshold = hotspots_only["street_tree_share"].median()
built_threshold = hotspots_only["built_up_share"].median()

def recommend_intervention(row):
    if not row["hotspot"]:
        return "No priority intervention"

    if row["green_space_share"] < green_threshold:
        return "Increase green space"

    elif row["street_tree_share"] < tree_threshold:
        return "Expand street trees and shade"

    elif (
        row["built_up_share"] > built_threshold
        and row["water_share"] < water_threshold
    ):
        return "Add cooling infrastructure in dense built-up areas"

    else:
        return "Targeted local cooling measures"

hotspot_data["recommended_intervention"] = hotspot_data.apply(
    recommend_intervention,
    axis=1
)

print(
    hotspot_data[hotspot_data["hotspot"]]
    ["recommended_intervention"]
    .value_counts()
)

print("\nRecommendations by city:")
print(
    pd.crosstab(
        hotspot_data.loc[
            hotspot_data["hotspot"], "city"
        ],
        hotspot_data.loc[
            hotspot_data["hotspot"], "recommended_intervention"
        ]
    )
)

recommended_intervention
Increase green space               477
Targeted local cooling measures    269
Expand street trees and shade      208
Name: count, dtype: int64

Recommendations by city:
recommended_intervention  Expand street trees and shade  Increase green space  \
city                                                                            
Amsterdam                                             3                    43   
Athina                                                8                     7   
Barcelona                                            21                    29   
Berlin                                               47                    64   
Budapest                                             17                    27   
Lisboa                                               31                     9   
Madrid                                               27                    35   
Milano                                                2                    89

In [142]:
print(hotspots_only["water_share"].describe())

print("\nWater threshold:")
print(water_threshold)

print("\nHotspots below water threshold:")
print((hotspots_only["water_share"] < water_threshold).sum())

print("\nHotspots with zero water:")
print((hotspots_only["water_share"] == 0).sum())

print("\nHotspots with water above zero:")
print((hotspots_only["water_share"] > 0).sum())

count    954.000000
mean       0.012183
std        0.031730
min        0.000000
25%        0.000000
50%        0.000000
75%        0.007790
max        0.256216
Name: water_share, dtype: float64

Water threshold:
0.0

Hotspots below water threshold:
0

Hotspots with zero water:
554

Hotspots with water above zero:
400


In [143]:
print("Zero-water hotspots:", (hotspots_only["water_share"] == 0).sum())

print(
    "\nZero water + low green:",
    (
        (hotspots_only["water_share"] == 0)
        & (hotspots_only["green_space_share"] < green_threshold)
    ).sum()
)

print(
    "Zero water + low street trees:",
    (
        (hotspots_only["water_share"] == 0)
        & (hotspots_only["street_tree_share"] < tree_threshold)
    ).sum()
)

print(
    "Zero water + high built-up:",
    (
        (hotspots_only["water_share"] == 0)
        & (hotspots_only["built_up_share"] > built_threshold)
    ).sum()
)

Zero-water hotspots: 554

Zero water + low green: 285
Zero water + low street trees: 309
Zero water + high built-up: 245


In [144]:
positive_water = hotspots_only.loc[
    hotspots_only["water_share"] > 0,
    "water_share"
]

print(positive_water.describe())
print("\nMedian positive water share:")
print(positive_water.median())

count    400.000000
mean       0.029055
std        0.043740
min        0.000006
25%        0.004261
50%        0.011675
75%        0.032957
max        0.256216
Name: water_share, dtype: float64

Median positive water share:
0.011674895520739394


In [ ]:
remaining_after_green_trees = hotspots_only[
    ~(
        (hotspots_only["green_space_share"] < green_threshold)
        | (hotspots_only["street_tree_share"] < tree_threshold)
    )
]

print("Remaining after Green and Trees:",
      len(remaining_after_green_trees))

print("Remaining with zero water:",
      (remaining_after_green_trees["water_share"] == 0).sum())

print("Remaining with water:",
      (remaining_after_green_trees["water_share"] > 0).sum())

### Check water recommendation after green space and street trees

Check how many hotspots remain after applying the green space and street tree rules, and how many of these hotspots have no water.

In [145]:
def recommend_intervention(row):
    if not row["hotspot"]:
        return "No priority intervention"

    if row["green_space_share"] < green_threshold:
        return "Increase green space"

    elif row["street_tree_share"] < tree_threshold:
        return "Expand street trees and shade"

    elif (
        row["built_up_share"] > built_threshold
        and row["water_share"] == 0
    ):
        return "Cooling measures in dense built-up areas"

    else:
        return "Targeted local cooling measures"

hotspot_data["recommended_intervention"] = hotspot_data.apply(
    recommend_intervention,
    axis=1
)

print(
    hotspot_data.loc[
        hotspot_data["hotspot"],
        "recommended_intervention"
    ].value_counts()
)

print("\nThresholds:")
print("Green:", round(green_threshold, 3))
print("Trees:", round(tree_threshold, 3))
print("Built-up:", round(built_threshold, 3))
print("Water:", round(water_threshold, 3))

recommended_intervention
Increase green space                        477
Expand street trees and shade               208
Targeted local cooling measures             152
Cooling measures in dense built-up areas    117
Name: count, dtype: int64

Thresholds:
Green: 0.07
Trees: 0.545
Built-up: 0.445
Water: 0.0


In [146]:
remaining_after_green_trees = hotspots_only[
    ~(
        (hotspots_only["green_space_share"] < green_threshold)
        | (hotspots_only["street_tree_share"] < tree_threshold)
    )
]

print("Remaining after Green and Trees:",
      len(remaining_after_green_trees))

print("Remaining with zero water:",
      (remaining_after_green_trees["water_share"] == 0).sum())

print("Remaining with water:",
      (remaining_after_green_trees["water_share"] > 0).sum())

Remaining after Green and Trees: 269
Remaining with zero water: 127
Remaining with water: 142


In [133]:
example_blocks = (
    hotspot_data[hotspot_data["hotspot"]]
    .groupby("recommended_intervention", group_keys=False)
    .head(1)
    [[
        "city",
        "green_space_share",
        "street_tree_share",
        "built_up_share",
        "water_share",
        "recommended_intervention"
    ]]
)

example_blocks

,city,green_space_share,street_tree_share,built_up_share,water_share,recommended_intervention
8,Amsterdam,0.019940,0.374834,0.331570,0.016573,Increase green space
67,Amsterdam,0.176616,0.658323,0.521889,0.221392,Targeted local cooling measures
90,Amsterdam,0.267248,0.537359,0.391499,0.251327,Expand street trees and shade
607,Amsterdam,0.138213,0.914895,0.734343,0.000000,Cooling measures in dense built-up areas


### Check overlap between zero water and dense built-up areas

Check how many remaining hotspots with no water also have a high built-up share before defining the final recommendation order.

In [147]:
zero_water_remaining = remaining_after_green_trees[
    remaining_after_green_trees["water_share"] == 0
]

print("Zero-water hotspots remaining:",
      len(zero_water_remaining))

print("Zero water + high built-up:",
      (zero_water_remaining["built_up_share"] > built_threshold).sum())

print("Zero water + not high built-up:",
      (zero_water_remaining["built_up_share"] <= built_threshold).sum())

Zero-water hotspots remaining: 127
Zero water + high built-up: 117
Zero water + not high built-up: 10


### Check low water among remaining hotspots

Check the water distribution of the remaining hotspots with some water before defining the final water recommendation rule.

In [148]:
remaining_with_water = remaining_after_green_trees[
    remaining_after_green_trees["water_share"] > 0
]

print(remaining_with_water["water_share"].describe())

count    142.000000
mean       0.038736
std        0.043670
min        0.000364
25%        0.008934
50%        0.023117
75%        0.049049
max        0.221392
Name: water_share, dtype: float64


In [149]:
def recommend_intervention(row):
    if not row["hotspot"]:
        return "No priority intervention"

    if row["green_space_share"] < green_threshold:
        return "Increase green space"

    elif row["street_tree_share"] < tree_threshold:
        return "Expand street trees and shade"

    elif (
        row["built_up_share"] > built_threshold
        and row["water_share"] == 0
    ):
        return "Cooling measures in dense built-up areas"

    elif row["water_share"] == 0:
        return "Add water features"

    else:
        return "Targeted local cooling measures"

In [150]:
hotspot_data["recommended_intervention"] = hotspot_data.apply(
    recommend_intervention,
    axis=1
)

print(
    hotspot_data.loc[
        hotspot_data["hotspot"],
        "recommended_intervention"
    ].value_counts()
)

recommended_intervention
Increase green space                        477
Expand street trees and shade               208
Targeted local cooling measures             142
Cooling measures in dense built-up areas    117
Add water features                           10
Name: count, dtype: int64


In [153]:
print(
    final_hotspots["recommended_intervention"]
    .value_counts()
)

recommended_intervention
Increase green space                        477
Expand street trees and shade               208
Targeted local cooling measures             152
Cooling measures in dense built-up areas    117
Name: count, dtype: int64


In [154]:
final_hotspots["recommended_intervention"] = hotspot_data.loc[
    final_hotspots.index,
    "recommended_intervention"
]

In [155]:
print(
    final_hotspots["recommended_intervention"]
    .value_counts()
)

recommended_intervention
Increase green space                        477
Expand street trees and shade               208
Targeted local cooling measures             142
Cooling measures in dense built-up areas    117
Add water features                           10
Name: count, dtype: int64


## City-Level Cooling Priorities

Local hotspot recommendations are summarised by study area to translate spatial evidence into clear urban cooling priorities.

In [134]:
city_recommendations = (
    hotspot_data[hotspot_data["hotspot"]]
    .groupby(["city", "recommended_intervention"])
    .size()
    .reset_index(name="hotspot_blocks")
)

city_main_recommendation = (
    city_recommendations
    .sort_values(
        ["city", "hotspot_blocks"],
        ascending=[True, False]
    )
    .drop_duplicates("city")
    .rename(
        columns={
            "recommended_intervention": "main_recommendation",
            "hotspot_blocks": "blocks_with_main_recommendation"
        }
    )
)

final_city_summary = city_hotspot_summary.merge(
    city_main_recommendation[
        [
            "city",
            "main_recommendation",
            "blocks_with_main_recommendation"
        ]
    ],
    on="city",
    how="left"
)

print(
    final_city_summary[
        [
            "city",
            "hotspot_blocks",
            "hotspot_share_pct",
            "population_exposed_pct",
            "main_recommendation",
            "blocks_with_main_recommendation"
        ]
    ]
    .sort_values("hotspot_share_pct", ascending=False)
    .round(2)
    .to_string(index=False)
)

     city  hotspot_blocks  hotspot_share_pct  population_exposed_pct                      main_recommendation  blocks_with_main_recommendation
   Milano             130              15.35                   53.52                     Increase green space                               89
Amsterdam              94              12.84                   54.72          Targeted local cooling measures                               47
Barcelona              87              12.31                   60.26                     Increase green space                               29
   Berlin             204               8.63                   62.36                     Increase green space                               64
    Praha             108               6.92                   29.83                     Increase green space                               65
     Wien             142               5.67                    6.69                     Increase green space                              109

In [135]:
# Final hotspot-level output
final_hotspots = hotspot_data[
    hotspot_data["hotspot"]
].copy()

# Final city-level summary
final_city_output = final_city_summary[
    [
        "city",
        "hotspot_blocks",
        "hotspot_share_pct",
        "hotspot_population",
        "population_exposed_pct",
        "main_recommendation",
        "blocks_with_main_recommendation"
    ]
].copy()

print("Final hotspot records:", len(final_hotspots))
print("Final city records:", len(final_city_output))

print("\nFinal city summary:")
print(
    final_city_output
    .sort_values("hotspot_share_pct", ascending=False)
    .round(2)
    .to_string(index=False)
)

Final hotspot records: 954
Final city records: 10

Final city summary:
     city  hotspot_blocks  hotspot_share_pct  hotspot_population  population_exposed_pct                      main_recommendation  blocks_with_main_recommendation
   Milano             130              15.35           2632508.0                   53.52                     Increase green space                               89
Amsterdam              94              12.84           1616346.0                   54.72          Targeted local cooling measures                               47
Barcelona              87              12.31           2925716.0                   60.26                     Increase green space                               29
   Berlin             204               8.63           3170534.0                   62.36                     Increase green space                               64
    Praha             108               6.92            701810.0                   29.83                     Incre

## Export Final Analysis Outputs

The final analytical datasets are exported for visualisation and presentation.

In [156]:
from pathlib import Path
import geopandas as gpd

# Output folder
output_dir = Path(
    "/Users/nedalgubran/Desktop/Project/"
    "Smart_Cities_Cooler_Futures/Data_Processed"
)

output_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Export city-level summary
# --------------------------------------------------

final_city_output.to_csv(
    output_dir / "smart_cities_city_summary.csv",
    index=False
)

# --------------------------------------------------
# 2. Add geometry to hotspot-level data
# --------------------------------------------------

final_hotspots_export = final_hotspots.merge(
    blocks_gdf[
        ["city", "block_x", "block_y", "geometry"]
    ],
    on=["city", "block_x", "block_y"],
    how="left"
)

final_hotspots_export = gpd.GeoDataFrame(
    final_hotspots_export,
    geometry="geometry",
    crs=blocks_gdf.crs
)

# --------------------------------------------------
# 3. Calculate latitude and longitude
# --------------------------------------------------

hotspot_centres = (
    final_hotspots_export.geometry
    .centroid
    .to_crs("EPSG:4326")
)

final_hotspots_export["longitude"] = hotspot_centres.x
final_hotspots_export["latitude"] = hotspot_centres.y

# --------------------------------------------------
# 4. Export hotspot-level dataset
# --------------------------------------------------

final_hotspots_export.drop(
    columns="geometry"
).to_csv(
    output_dir / "smart_cities_hotspots.csv",
    index=False
)

# --------------------------------------------------
# 5. Validation
# --------------------------------------------------

print("Export complete")
print("City rows:", len(final_city_output))
print("Hotspot rows:", len(final_hotspots_export))
print(
    "Missing coordinates:",
    final_hotspots_export["latitude"].isna().sum()
)

print("\nFiles created:")
print("smart_cities_city_summary.csv")
print("smart_cities_hotspots.csv")

Export complete
City rows: 10
Hotspot rows: 954
Missing coordinates: 0

Files created:
smart_cities_city_summary.csv
smart_cities_hotspots.csv


In [158]:
check_export = pd.read_csv(
    output_dir / "smart_cities_hotspots.csv"
)

print(
    check_export["recommended_intervention"]
    .value_counts()
)

recommended_intervention
Increase green space                        477
Expand street trees and shade               208
Targeted local cooling measures             142
Cooling measures in dense built-up areas    117
Add water features                           10
Name: count, dtype: int64


In [159]:
from pathlib import Path

output_dir = Path(
    "/Users/nedalgubran/Desktop/Project/"
    "Smart_Cities_Cooler_Futures/Data_Processed"
)

output_dir.mkdir(parents=True, exist_ok=True)

final_hotspots_export.drop(columns="geometry").to_csv(
    output_dir / "smart_cities_hotspots.csv",
    index=False
)

print("Saved to:")
print(output_dir / "smart_cities_hotspots.csv")

print("\nRecommendations:")
print(
    final_hotspots_export["recommended_intervention"]
    .value_counts()
)

Saved to:
/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/Data_Processed/smart_cities_hotspots.csv

Recommendations:
recommended_intervention
Increase green space                        477
Expand street trees and shade               208
Targeted local cooling measures             142
Cooling measures in dense built-up areas    117
Add water features                           10
Name: count, dtype: int64


In [160]:
final_path = (
    "/Users/nedalgubran/Desktop/Project/"
    "Smart_Cities_Cooler_Futures/data_processed/"
    "smart_cities_hotspots_FINAL.csv"
)

final_hotspots_export.drop(columns="geometry").to_csv(
    final_path,
    index=False
)

check = pd.read_csv(final_path)

print(final_path)
print(check["recommended_intervention"].value_counts())

/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/data_processed/smart_cities_hotspots_FINAL.csv
recommended_intervention
Increase green space                        477
Expand street trees and shade               208
Targeted local cooling measures             142
Cooling measures in dense built-up areas    117
Add water features                           10
Name: count, dtype: int64


## Final Decision-Support Outputs

The final outputs combine statistical evidence, local heat hotspots, population exposure and evidence-based cooling recommendations.

These outputs provide the basis for the final Tableau visualisations and project conclusions.

# Decision-Support Stage

This stage is deliberately placed after statistical evidence.

A final prioritisation model may combine heat burden, exposure, and cooling capacity, but its interpretation must remain explicit:

- Statistical tests identify supported associations with heat.
- Descriptive indicators characterise each city's urban profile.
- A Priority Score, if retained, is a transparent decision-support index, not a causal model.
- Recommendations must be traceable to observed conditions and evidence-supported intervention logic.

The final score specification and recommendations are completed only after the spatial hypothesis tests have been validated.


# Final Deliverables

The completed notebook will export:

1. A statistical evidence table for the tested hypotheses.
2. A city-level decision-support table for SQL and Tableau.
3. Evidence-supported recommendation fields.
4. A concise methodology and limitations summary for the final presentation.

Presentation logic:

**Problem → Data → Hypotheses → Statistical Evidence → Priority → Action → Conclusion**


In [137]:
import pandas as pd

regression_results = pd.DataFrame({
    "factor": [
        "Green Space",
        "Water",
        "Built-up Share",
        "Street Trees",
        "Mean Building Height",
        "Building Density"
    ],
    "effect": [
        -0.61,
        -0.77,
         0.12,
        -0.10,
         0.10,
         0.00
    ],
    "unit": [
        "per +10 percentage points",
        "per +10 percentage points",
        "per +10 percentage points",
        "per +10 percentage points",
        "per +1 metre",
        "No independent association"
    ],
    "significance": [
        "p < 0.001",
        "p < 0.001",
        "p = 0.001",
        "p < 0.001",
        "p < 0.001",
        "p = 0.94"
    ]
})

regression_results.to_csv(
    "smart_cities_regression_results.csv",
    index=False
)

regression_results

,factor,effect,unit,significance
0,Green Space,-0.61,per +10 percentage points,p < 0.001
1,Water,-0.77,per +10 percentage points,p < 0.001
2,Built-up Share,0.12,per +10 percentage points,p = 0.001
3,Street Trees,-0.10,per +10 percentage points,p < 0.001
4,Mean Building Height,0.10,per +1 metre,p < 0.001
5,Building Density,0.00,No independent association,p = 0.94


In [138]:
import os
print(os.path.abspath("smart_cities_regression_results.csv"))

/Users/nedalgubran/Desktop/Project/Smart_Cities_Cooler_Futures/Notebooks/smart_cities_regression_results.csv


In [139]:
import os
print(os.listdir(".."))

['background', 'Streamlit', '.DS_Store', 'data_processed', 'tableau', 'simple_app', 'README.md', 'download_lst.py', '.gitignore', 'data_raw', '.git', 'Notebooks', 'SQL']


In [140]:
regression_results.to_csv("../data_processed/smart_cities_regression_results.csv", index=False)

In [141]:
example_blocks

,city,green_space_share,street_tree_share,built_up_share,water_share,recommended_intervention
8,Amsterdam,0.019940,0.374834,0.331570,0.016573,Increase green space
67,Amsterdam,0.176616,0.658323,0.521889,0.221392,Targeted local cooling measures
90,Amsterdam,0.267248,0.537359,0.391499,0.251327,Expand street trees and shade
607,Amsterdam,0.138213,0.914895,0.734343,0.000000,Cooling measures in dense built-up areas
